# LLM-анализ блока «Мотивация клиента на оплату» в диалогах взыскания

## Постановка задачи

Нужно разработать решение на основе prompt'ов для LLM, которое по тексту диалога оператора с клиентом и сроку просрочки определяет, выполнен ли блок **«Мотивация клиента на оплату»**.

Оценка выполняется по правилам из ТЗ: учитываются категория клиента, согласие или отказ от оплаты, озвученные последствия, допустимость формулировок и исключения.

## Категории и статус клиента

Категория клиента определяется по сроку просрочки задолженности.

| Категория | Срок просрочки |
|---|---:|
| A | 1–30 дней |
| Б | 31–45 дней |

Статус клиента определяется по его реакции на предложение оператора об оплате.

| Статус клиента | Описание |
|---|---|
| Согласен оплатить | Клиент согласен оплатить в предложенный оператором срок |
| Не согласен оплатить | Клиент отказывается, торгуется или просит отсрочку |

Короткий ответ вроде «угу» может считаться согласием, если он дан как подтверждение оплаты в ответ на вопрос оператора.

## Правила оценки по категориям

| Категория | Статус клиента | Условие выполнения критерия |
|---|---|---|
| A | Согласен | 1 последствие неоплаты, констатация действующего последствия или позитивное уведомление об отмене последствий |
| A | Не согласен | 2 последствия неоплаты или констатация действующих последствий |
| Б | Согласен | 1 последствие из категории A |
| Б | Не согласен | 1 последствие из категории A + 1 последствие из категории Б |

## Последствия

| Категория A | Категория Б, дополнительный список |
|---|---|
| ухудшение кредитной истории | требование полного досрочного погашения |
| штрафы / неустойки | ст. 811 ГК РФ |
| звонки | продажа долга |
| выезд сотрудника | передача в работу коллекторов |
| продажа долга |  |
| передача в работу коллекторов |  |

Для клиентов категории A ошибкой периода считается использование последствий, применимых только к категории Б, например требования полного досрочного погашения или ссылки на ст. 811 ГК РФ.

## Допустимые и запрещённые формулировки

Оператор должен использовать допустимые формулировки: **«вправе»**, **«может»**, **«возможно»**.

| Разрешено | Запрещено |
|---|---|
| вероятностные формулировки: «может», «вправе», «возможно» | гарантии наступления последствий: «будет», «передадим» |
| точные суммы | точные даты будущих последствий |
| констатация уже начисляемых штрафов | формулировки вида «завтра передадим» |
| констатация ухудшающейся кредитной истории |  |
| констатация поступающих звонков |  |

## Исключения

Критерий считается выполненным автоматически, если в диалоге упоминается одно из обстоятельств:

- банкротство;
- ЧС;
- военные действия;
- тюрьма;
- армия;
- мошенничество;
- смерть плательщика;
- инвалидность;
- лечение.

Также критерий считается выполненным автоматически, если оператор не попрощался в конце диалога.

## Примеры из ТЗ

| Кейс | Ситуация | Ожидаемый результат | Причина |
|---|---|---|---|
| Согласие + позитивное уведомление | Просрочка 10 дней, клиент: «Сегодня оплачу». Оператор: «Как только платеж поступит, звонки прекратятся» | Выполнено | Есть позитивное уведомление об отмене последствия |
| Нет мотивации | Клиент согласен. Оператор: «Ждём платежа, себя не подводите» | Не выполнено | Не озвучены последствия |
| Ошибка периода | Просрочка 10 дней, оператор говорит про полное досрочное погашение и передачу долга коллекторам | Не выполнено | Для категории A использовано последствие категории Б |
| Несогласие + 2 последствия | Просрочка 22 дня, клиент не может оплатить сегодня, оператор говорит про штрафы и звонки | Выполнено | Озвучены 2 последствия |
| Запрещённая форма | «Если не оплатите до 13 апреля, мы передадим дело коллекторам» | Не выполнено | Есть точная дата и гарантия |
| Исключение | Клиент сообщает о лечении | Выполнено | Сработало исключение |
| «Угу» как согласие | Клиент отвечает «угу» на подтверждение оплаты | Выполнено | Ответ засчитан как согласие |

## Целевые признаки для оценки

| Признак | Зачем нужен |
|---|---|
| Срок просрочки | Определение категории клиента |
| Статус клиента | Выбор правила для согласного / несогласного клиента |
| Последствия, озвученные оператором | Проверка наличия мотивации |
| Категория последствия | Проверка соответствия категории клиента |
| Формулировка последствия | Проверка запрета гарантий и точных дат |
| Исключения | Автоматическое выполнение критерия |
| Завершение диалога | Проверка признака прерванного диалога |

## Baseline-подход

В baseline используется один монолитный prompt и один вызов LLM:

```text
текст диалога + срок просрочки → prompt → JSON-оценка
```

Внутри prompt модель должна последовательно:
1. проверить исключения;
2. определить категорию клиента;
3. определить статус клиента;
4. найти последствия;
5. проверить допустимость формулировок;
6. сопоставить признаки с правилами категории;
7. вернуть итоговое решение.

Такой подход выбран как стартовый: он проще в реализации, быстрее проверяется и достаточен для первичной оценки качества prompt'а.

| Плюсы | Ограничения |
|---|---|
| Один вызов модели | Сложнее локализовать ошибку внутри reasoning |
| Простая реализация | Большой prompt может быть менее устойчив на сложных кейсах |
| Удобно быстро тестировать | Часть проверок потенциально можно вынести в детерминированную логику |

## Допущения baseline

- Prompt'ы написаны на русском языке, так как диалоги и правила из ТЗ русскоязычные.
- Для некоторых multilingual-моделей английские system-инструкции могут быть стабильнее; это можно проверить отдельно при адаптации решения под конкретную LLM.
- Целевая LLM заранее неизвестна, поэтому в production prompt strategy нужно подбирать под конкретную модель.
- Качество оценки зависит от качества текстовой расшифровки диалога.
- Спорные случаи могут требовать ручной проверки или дополнительной валидации.

# Реализация baseline

In [2]:
import json
from typing import Any, Dict, List, Optional

## System prompt

Ниже задаётся основной prompt с правилами оценки из ТЗ.

In [3]:
SYSTEM_PROMPT = """
Ты оцениваешь качество мотивации клиента на оплату задолженности.
Следуй инструкции пользователя.
Верни только валидный JSON.
""".strip()

## User prompt template

В user prompt передаются только переменные конкретного кейса: срок просрочки и текст диалога.

In [38]:
overdue_for_A = "категория A: 1–30 дней просрочки"
overdue_for_B = "категория Б: 31–45 дней просрочки"

payment_true = """
клиент согласен оплатить задолженность.
Согласие может быть выражено явно: «оплачу», «внесу платёж», «закрою сегодня», «да, подтверждаю».
Оператор может не называть конкретный срок, если клиент сам заявил намерение оплатить.
""".strip()

payment_false = """
клиент отказывается, торгуется, просит отсрочку, говорит «не могу», «постараюсь», «возможно»,
или не подтверждает оплату уверенно.
""".strip()

In [39]:
eval_rules_A = """
- если клиент согласен оплатить: должно быть озвучено 1 последствие неоплаты,
  констатация действующего последствия или позитивное уведомление об отмене последствий;
- если клиент не согласен оплатить: должно быть озвучено 2 последствия неоплаты
  или констатация действующих последствий.
""".strip()

eval_rules_B = """
- если клиент согласен оплатить: должно быть озвучено 1 последствие именно из категории A;
- если клиент не согласен оплатить: должно быть озвучено минимум 1 последствие из категории A
  и минимум 1 дополнительное последствие из категории Б.

Важно:
- для согласного клиента категории Б недостаточно озвучить только дополнительное последствие категории Б;
- дополнительные последствия категории Б используются как обязательное дополнение только если клиент не согласен оплатить.
""".strip()

In [60]:
motivation_forms = """
Мотивацией считается одно из трёх:

1. Последствие неоплаты:
   например штрафы / неустойки, звонки, ухудшение кредитной истории, выезд сотрудника,
   продажа долга, передача в работу коллекторов, требование полного досрочного погашения,
   ст. 811 ГК РФ.

2. Констатация действующего последствия:
   например уже начисляются штрафы, уже поступают звонки, уже ухудшается кредитная история.

3. Позитивное уведомление об отмене последствия после оплаты:
   например после оплаты звонки прекратятся,
   после оплаты информация по кредитной истории обновится,
   после поступления платежа взаимодействие по задолженности будет прекращено.

Позитивное уведомление об отмене последствия после оплаты засчитывается как мотивация.
""".strip()

In [40]:
consequences_A = """
- ухудшение кредитной истории;
- штрафы / неустойки;
- звонки;
- выезд сотрудника;
- продажа долга;
- передача в работу коллекторов.
""".strip()

consequences_B = """
- требование полного досрочного погашения;
- ст. 811 ГК РФ;
- продажа долга;
- передача в работу коллекторов.
""".strip()

In [41]:
allowed_words = """
- вправе;
- может;
- возможно.
""".strip()

not_allowed_words = """
- точные даты наступления последствий;
- гарантии будущих последствий, например: «будет», «завтра передадим», «мы передадим»;
- утвердительные формулировки будущих последствий.
""".strip()

In [42]:
exceptional_situations = """
- банкротство;
- ЧС;
- военные действия;
- тюрьма;
- армия;
- мошенничество;
- смерть плательщика;
- инвалидность;
- лечение.
""".strip()

auto_pass_rules = """
Критерий считается выполненным автоматически, если:
- в диалоге найдено исключение;
- оператор не попрощался в конце диалога.
""".strip()

In [43]:
role_rules = """
- последствия и нарушения ищи только в репликах оператора;
- согласие или отказ клиента определяй только по репликам клиента;
- не засчитывай последствия, которые озвучил клиент;
- не придумывай последствия, которых нет в репликах оператора;
- не используй списки правил как найденные последствия в диалоге.
""".strip()

In [44]:
fail_rules = """
Критерий считается невыполненным, если:
- оператор не озвучил нужное количество последствий для категории клиента и статуса клиента;
- оператор использовал запрещённую формулировку;
- оператор назвал точную дату наступления последствия;
- оператор гарантировал будущее последствие;
- клиент относится к категории A, а оператор озвучил требование полного досрочного погашения или ст. 811 ГК РФ.
""".strip()

In [61]:
restrictions = """
- оценивай только блок «Мотивация клиента на оплату»;
- не оценивай другие аспекты диалога;
- учитывай только текст диалога и срок просрочки;
- не придумывай факты, последствия, согласие клиента или нарушения;
- ответ «угу» может считаться согласием, если он подтверждает оплату;
- поле detected_consequences заполняй только по явно найденным репликам оператора;
- если последствий нет, detected_consequences должен быть пустым списком [];
- если оператор сказал, что после оплаты последствие прекратится или ситуация улучшится, засчитывай это как мотивацию;
- позитивное уведомление записывай в detected_consequences с type="позитивное уведомление".
""".strip()

In [46]:
decision_steps = """
Порядок проверки:

1. Определи категорию клиента только по сроку просрочки.

2. Проверь автоматический зачёт:
   - исключение;
   - оператор не попрощался в конце диалога.

3. Определи статус клиента:
   - согласен оплатить;
   - не согласен оплатить.

4. Найди последствия, озвученные оператором.

5. Найди нарушения в репликах оператора.

6. Прими решение:
   - сначала проверь автоматический зачёт;
   - затем проверь нарушения;
   - затем проверь, хватает ли последствий по категории клиента и статусу клиента.
""".strip()

## Формат ответа модели

Модель должна возвращать валидный JSON, чтобы результат можно было автоматически проверить и сохранить.

```json
{
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [
    {
      "text": "После оплаты звонки прекратятся",
      "type": "звонки",
      "category": "A"
    }
  ],
  "violations": [],
  "explanation": "Клиент согласился оплатить, оператор озвучил позитивное уведомление об отмене последствия."
}
```

### Основные поля ответа

| Поле | Назначение |
|---|---|
| criterion_met | Итоговый результат проверки |
| decision | Текстовый итог проверки |
| client_category | Категория клиента |
| client_status | Статус клиента |
| exception_detected | Было ли найдено исключение |
| interrupted_dialog | Был ли диалог прерван |
| detected_consequences | Найденные последствия |
| violations | Найденные нарушения |
| explanation | Объяснение итогового решения |

In [47]:
output_schema = """
{
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [],
  "violations": [],
  "explanation": "Краткое объяснение в 1 предложении без прямых цитат из диалога."
}
""".strip()

In [62]:
def build_user_prompt(dialogue: str, overdue_days: int) -> str:
    return f"""
Оцени, насколько качественно оператор мотивирует клиента на оплату задолженности.

Входные данные:
Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории клиентов:
- {overdue_for_A}
- {overdue_for_B}

Как определить согласие клиента:
Согласен:
{payment_true}

Не согласен:
{payment_false}

Правила оценки категории A:
{eval_rules_A}

Правила оценки категории Б:
{eval_rules_B}

Последствия категории A:
{consequences_A}

Дополнительные последствия категории Б:
{consequences_B}

Формы мотивации:
{motivation_forms}

Допустимые формулировки оператора:
{allowed_words}

Недопустимые формулировки оператора:
{not_allowed_words}

Исключения:
{exceptional_situations}

Правила автоматического зачёта:
{auto_pass_rules}

Правила ролей:
{role_rules}

Правила незачёта:
{fail_rules}

Общие ограничения:
{restrictions}

Порядок проверки:
{decision_steps}

Верни только валидный JSON без markdown-разметки и дополнительных комментариев.

Формат ответа:
{output_schema}
""".strip()

## Вызов LLM

In [7]:
from openai import OpenAI, APITimeoutError, APIConnectionError, RateLimitError, APIStatusError

In [8]:
import os
import time

In [9]:
import getpass
API_KEY = getpass.getpass("Enter API key: ")
BASE_URL = getpass.getpass("Enter base URL: ")

Enter API key: ··········
Enter base URL: ··········


In [10]:
MODEL_NAME = "gpt-4o-mini"

In [96]:
client = OpenAI(api_key=API_KEY,
                base_url=BASE_URL,
                timeout=15.0,
                max_retries=3,)

In [97]:
def evaluate_dialogue(
    dialogue: str,
    overdue_days: int,
    model: str = MODEL_NAME,
    temperature: float = 0.0,
    max_tokens: int = 700,
    max_attempts: int = 3,
    verbose: bool = True,
) -> Dict[str, Any]:
    user_prompt = build_user_prompt(
        dialogue=dialogue,
        overdue_days=overdue_days,
    )

    start_time = time.perf_counter()
    last_error = None
    raw_content = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )

            raw_content = response.choices[0].message.content
            parsed_result = json.loads(raw_content)

            elapsed_time = time.perf_counter() - start_time

            if verbose:
                print("Model:", model)
                print("Base URL:", BASE_URL)
                print("Prompt length:", len(user_prompt))
                print(f"Attempts used: {attempt}")
                print(f"Elapsed time: {elapsed_time:.2f} sec")
                print("Raw answer:\n", raw_content)

            parsed_result["_elapsed_time_sec"] = round(elapsed_time, 2)
            parsed_result["_prompt_length"] = len(user_prompt)
            parsed_result["_model"] = model
            parsed_result["_attempts_used"] = attempt

            return parsed_result

        except Exception as error:
            last_error = error

            if verbose:
                print(f"Attempt {attempt}/{max_attempts} failed: {type(error).__name__}")

    raise RuntimeError(
        f"Не удалось получить корректный ответ после {max_attempts} попыток.\n"
        f"Последняя ошибка: {type(last_error).__name__}: {last_error}\n\n"
        f"Последний raw_content:\n{raw_content}"
    )

In [65]:
def print_evaluation_result(result: Dict[str, Any]) -> None:
    print("=== Результат оценки ===")
    print(f"Итог: {result.get('decision')}")
    print(f"Критерий выполнен: {result.get('criterion_met')}")
    print(f"Категория клиента: {result.get('client_category')}")
    print(f"Статус клиента: {result.get('client_status')}")
    print(f"Исключение найдено: {result.get('exception_detected')}")
    print(f"Прерванный диалог: {result.get('interrupted_dialog')}")

    print("\n=== Найденные последствия ===")
    consequences = result.get("detected_consequences", [])
    if consequences:
        for idx, consequence in enumerate(consequences, start=1):
            print(f"{idx}. Текст: {consequence.get('text')}")
            print(f"   Тип: {consequence.get('type')}")
            print(f"   Категория: {consequence.get('category')}")
    else:
        print("Не найдены")

    print("\n=== Нарушения ===")
    violations = result.get("violations", [])
    if violations:
        for idx, violation in enumerate(violations, start=1):
            if isinstance(violation, dict):
                print(f"{idx}. Текст: {violation.get('text')}")
                print(f"   Тип: {violation.get('type')}")
                print(f"   Причина: {violation.get('reason')}")
            else:
                print(f"{idx}. {violation}")
    else:
        print("Не найдены")

    print("\n=== Объяснение ===")
    print(result.get("explanation"))

# Тестирование baseline

Для проверки baseline используется небольшой набор кейсов, покрывающих основные правила из ТЗ: успешные сценарии, отсутствие мотивации, ошибку периода, запрещённые формулировки, исключения и неявное согласие.

Для каждого кейса сравнивается ожидаемый результат с ответом модели.

## Первый тестовый диалог

In [66]:
test_dialogue = """
Оператор: Подтверждаете оплату до пятницы?
Клиент: Угу.
Оператор: После оплаты звонки прекратятся.
Оператор: До свидания.
""".strip()

In [67]:
result = evaluate_dialogue(
    dialogue=test_dialogue,
    overdue_days=10,
)

Model: gpt-4o-mini
Base URL: https://api.artemox.com/v1
Prompt length: 5575
Attempts used: 1
Elapsed time: 12.88 sec
Raw answer:
 {
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [
    {
      "type": "позитивное уведомление",
      "text": "После оплаты звонки прекратятся."
    }
    ],
  "violations": [],
  "explanation": "Клиент выразил согласие оплатить, оператор упомянул позитивное уведомление об отмене последствий после оплаты, что соответствует требованиям для категории A."
}


In [68]:
print_evaluation_result(result)

=== Результат оценки ===
Итог: выполнено
Критерий выполнен: True
Категория клиента: A
Статус клиента: согласен оплатить
Исключение найдено: False
Прерванный диалог: False

=== Найденные последствия ===
1. Текст: После оплаты звонки прекратятся.
   Тип: позитивное уведомление
   Категория: None

=== Нарушения ===
Не найдены

=== Объяснение ===
Клиент выразил согласие оплатить, оператор упомянул позитивное уведомление об отмене последствий после оплаты, что соответствует требованиям для категории A.


## Несколько тестов

### Тестовые примеры

In [69]:
test_cases_tz = [
    {
        "case_id": "tz_01_A_agree_positive_notification",
        "description": "Пример 1 из ТЗ: категория A, клиент согласен, позитивное уведомление",
        "overdue_days": 10,
        "expected": True,
        "dialogue": """
Клиент: Сегодня оплачу.
Оператор: Как только платеж поступит, звонки прекратятся.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_02_A_agree_no_motivation",
        "description": "Пример 2 из ТЗ: категория A, клиент согласен, нет мотивации",
        "overdue_days": 10,
        "expected": False,
        "dialogue": """
Клиент: Сегодня оплачу.
Оператор: Ждём платежа, себя не подводите.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_03_A_wrong_period",
        "description": "Пример 3 из ТЗ: категория A, ошибка периода",
        "overdue_days": 10,
        "expected": False,
        "dialogue": """
Оператор: Банк может потребовать полное досрочное погашение и передать долг коллекторам.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_04_A_disagree_two_consequences",
        "description": "Пример 4 из ТЗ: категория A, клиент не согласен, 2 последствия",
        "overdue_days": 22,
        "expected": True,
        "dialogue": """
Клиент: Не могу сегодня.
Оператор: Банк вправе ежедневно начислять штрафы — это переплата, долг растёт.
Оператор: Также могут быть звонки вам и контактным лицам, создавая неудобства.
Оператор: Как планируете решать?
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_05_A_forbidden_assertive_wording",
        "description": "Пример 5 из ТЗ: точная дата последствия и гарантия",
        "overdue_days": 15,
        "expected": False,
        "dialogue": """
Оператор: Если не оплатите до 13 апреля, мы передадим дело коллекторам.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_06_exception_treatment",
        "description": "Пример 6 из ТЗ: исключение — лечение",
        "overdue_days": 18,
        "expected": True,
        "dialogue": """
Клиент: У меня инсульт, я на лечении.
Оператор: Понимаю, выздоравливайте.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_08_implicit_agreement_ugu",
        "description": "Пример 8 из ТЗ: «угу» как согласие",
        "overdue_days": 10,
        "expected": True,
        "dialogue": """
Оператор: Подтверждаете оплату до пятницы?
Клиент: Угу.
Оператор: Отлично, после оплаты обновим кредитную историю.
Оператор: До свидания.
""".strip(),
    },
]

In [70]:
manual_case_tz_words = {
    "case_id": "manual_01_tz_words_complex",
    "description": "Ручной длинный кейс: формулировки близки к ТЗ, проверка нескольких признаков",
    "overdue_days": 22,
    "expected": True,
    "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность, нужно обсудить оплату.
Клиент: Сегодня не могу оплатить, денег нет.
Оператор: Правильно понимаю, что сегодня оплатить не получится?
Клиент: Да, сегодня не смогу.
Оператор: Банк вправе ежедневно начислять штрафы — это переплата, долг растёт.
Оператор: Также могут быть звонки вам и контактным лицам, создавая неудобства.
Оператор: Как планируете решать вопрос с оплатой?
Клиент: Постараюсь внести платёж завтра или послезавтра.
Оператор: Хорошо, ожидаем оплату. До свидания.
""".strip(),
}

In [71]:
manual_case_synonyms = {
    "case_id": "manual_02_synonyms_complex",
    "description": "Ручной длинный кейс: естественные синонимы вместо точных формулировок",
    "overdue_days": 36,
    "expected": True,
    "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченной задолженности. Когда сможете внести платёж?
Клиент: Сейчас не готов платить, нужна отсрочка хотя бы на пару недель.
Оператор: То есть в предложенный срок оплату подтвердить не можете?
Клиент: Да, пока не могу.
Оператор: В таком случае банк может продолжить начисление неустойки, и сумма задолженности станет больше.
Оператор: Также банк вправе потребовать вернуть всю сумму задолженности раньше установленного графика.
Клиент: Понял, но сейчас всё равно денег нет.
Оператор: Какой срок оплаты вы можете назвать сейчас?
Клиент: Попробую решить вопрос в течение недели.
Оператор: Спасибо за информацию. До свидания.
""".strip(),
}

### Второй тест baseline

In [72]:
test_cases = test_cases_tz + [manual_case_tz_words,
                              manual_case_synonyms,]

In [73]:
import pandas as pd
from tqdm.auto import tqdm

In [74]:
def run_evaluation(test_cases: list[dict],
                   verbose: bool = False,) -> tuple[pd.DataFrame, float]:

    results = []

    total_start_time = time.perf_counter()

    for case in tqdm(test_cases, desc="Evaluating dialogues"):
        result = evaluate_dialogue(dialogue=case["dialogue"],
                                   overdue_days=case["overdue_days"],
                                   verbose=verbose,)

        results.append({"case_id": case["case_id"],
                        "description": case["description"],
                        "overdue_days": case["overdue_days"],
                        "expected": case["expected"],
                        "predicted": result.get("criterion_met"),
                        "is_correct": result.get("criterion_met") == case["expected"],
                        "decision": result.get("decision"),
                        "client_category": result.get("client_category"),
                        "client_status": result.get("client_status"),
                        "exception_detected": result.get("exception_detected"),
                        "interrupted_dialog": result.get("interrupted_dialog"),
                        "detected_consequences": result.get("detected_consequences"),
                        "violations": result.get("violations"),
                        "elapsed_time_sec": result.get("_elapsed_time_sec"),
                        "explanation": result.get("explanation"),})

    total_elapsed_time = time.perf_counter() - total_start_time

    results_df = pd.DataFrame(results)

    return results_df, total_elapsed_time

In [75]:
results_df, total_elapsed_time = run_evaluation(test_cases=test_cases,
                                                verbose=False,)

results_df

Evaluating dialogues:   0%|          | 0/9 [00:00<?, ?it/s]

,case_id,description,overdue_days,expected,predicted,is_correct,decision,client_category,client_status,exception_detected,interrupted_dialog,detected_consequences,violations,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",10,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'последствие неоплаты', 'text': 'Как...",[],11.66,"Клиент выразил намерение оплатить, оператор по..."
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",10,False,True,False,выполнено,A,согласен оплатить,False,False,[],[],11.87,"Клиент выразил намерение оплатить, а оператор ..."
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",10,False,True,False,выполнено,A,согласен оплатить,False,False,[],[],61.10,Категория A и статус клиента совпадают с согла...
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",22,True,True,True,выполнено,A,не согласен оплатить,False,False,[],[],11.84,Срок просрочки 22 дня ставит клиента в категор...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,15,False,True,False,выполнено,A,согласен оплатить,False,False,"[{'type': 'последствие неоплаты', 'text': 'Есл...",[],12.21,Срок просрочки 15 дней относится к категории A...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,18,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'п...",[],12.79,Клиенту присвоена категория A; оператор не поп...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,10,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'п...",[],12.58,"Клиент согласен оплатить, оператор упомянул по..."
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",22,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'п...",[оператор озвучил последствия неоплаты до подт...,28.16,"Клиент заявил намерение оплатить, оператор ука..."
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,36,True,True,True,выполнено,Б,не согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'statement...",[],11.83,Клиент на 36-й день просрочки относится к кате...


In [76]:
summary_df = results_df[["case_id","description","expected","predicted","is_correct",
                         "decision","elapsed_time_sec","explanation",]]

summary_df

,case_id,description,expected,predicted,is_correct,decision,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",True,True,True,выполнено,11.66,"Клиент выразил намерение оплатить, оператор по..."
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",False,True,False,выполнено,11.87,"Клиент выразил намерение оплатить, а оператор ..."
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,True,False,выполнено,61.10,Категория A и статус клиента совпадают с согла...
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",True,True,True,выполнено,11.84,Срок просрочки 22 дня ставит клиента в категор...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,True,False,выполнено,12.21,Срок просрочки 15 дней относится к категории A...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,True,True,True,выполнено,12.79,Клиенту присвоена категория A; оператор не поп...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,True,True,True,выполнено,12.58,"Клиент согласен оплатить, оператор упомянул по..."
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",True,True,True,выполнено,28.16,"Клиент заявил намерение оплатить, оператор ука..."
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,True,True,True,выполнено,11.83,Клиент на 36-й день просрочки относится к кате...


In [77]:
print("=== Итоги тестирования ===")
print(f"Всего кейсов: {len(results_df)}")
print(f"Корректных ответов: {results_df['is_correct'].sum()}")
print(f"Accuracy: {results_df['is_correct'].mean():.2%}")
print(f"Общее время: {total_elapsed_time:.2f} сек.")
print(f"Среднее время на кейс: {results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {results_df['elapsed_time_sec'].max():.2f} сек.")

=== Итоги тестирования ===
Всего кейсов: 9
Корректных ответов: 6
Accuracy: 66.67%
Общее время: 174.07 сек.
Среднее время на кейс: 19.34 сек.
Минимальное время: 11.66 сек.
Максимальное время: 61.10 сек.


In [78]:
incorrect_df = results_df[~results_df["is_correct"]]

incorrect_df[["case_id","description","expected","predicted","decision",
              "detected_consequences","violations","explanation",]]

,case_id,description,expected,predicted,decision,detected_consequences,violations,explanation
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",False,True,выполнено,[],[],"Клиент выразил намерение оплатить, а оператор ..."
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,True,выполнено,[],[],Категория A и статус клиента совпадают с согла...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,True,выполнено,"[{'type': 'последствие неоплаты', 'text': 'Есл...",[],Срок просрочки 15 дней относится к категории A...


In [79]:
for _, row in incorrect_df.iterrows():
    print("=" * 80)
    print("CASE:", row["case_id"])
    print("EXPECTED:", row["expected"])
    print("PREDICTED:", row["predicted"])
    print("\nEXPLANATION:")
    print(row["explanation"])
    print()

CASE: tz_02_A_agree_no_motivation
EXPECTED: False
PREDICTED: True

EXPLANATION:
Клиент выразил намерение оплатить, а оператор не назвал конкретных последствий неоплаты и не упоминал дату, что соответствует категории A при согласии клиента.

CASE: tz_03_A_wrong_period
EXPECTED: False
PREDICTED: True

EXPLANATION:
Категория A и статус клиента совпадают с согласие на оплату, однако оператор не озвучил ни одного последствия из категории A.

CASE: tz_05_A_forbidden_assertive_wording
EXPECTED: False
PREDICTED: True

EXPLANATION:
Срок просрочки 15 дней относится к категории A; оператор упомянул одно последствие неоплаты и сделал уведомление об передаче коллекторской службе, что соответствует требованиям категории A при согласии клиента.



### Анализ первого прогона baseline

Baseline-решение использует один prompt и один вызов LLM для оценки диалога. Такой подход прост в реализации и позволяет быстро получить первичную оценку качества мотивации оператора.

На тестовых примерах baseline показал, что модель в целом понимает структуру задачи, но допускает системные ошибки:

- путает согласие клиента на оплату с качественной мотивацией со стороны оператора;
- может считать критерий выполненным, даже если оператор не озвучил последствие, констатацию действующего последствия или позитивное уведомление;
- не всегда блокирует критерий при ошибке периода;
- не всегда блокирует критерий при запрещённых утвердительных формулировках.

Главная проблема baseline: модель оценивает общий исход диалога, а не строго выполнение блока «Мотивация клиента на оплату» со стороны оператора.

Далее prompt будет доработан точечно: нужно явно разделить согласие клиента и мотивацию оператора.

## Модификация промпта

In [80]:
evaluation_focus = """
Важно: оценивается не сам факт согласия клиента на оплату, а качество мотивации со стороны оператора.

Клиент может согласиться оплатить, но критерий всё равно будет не выполнен,
если оператор не озвучил требуемую мотивацию.

Мотивация оператора засчитывается только если оператор сделал хотя бы одно из действий:
- озвучил последствие неоплаты;
- констатировал уже действующее последствие;
- дал позитивное уведомление об отмене последствия после оплаты.

Фразы оператора вроде «ждём платёж», «ожидаем оплату», «себя не подводите»,
«не затягивайте» не являются мотивацией.

Если клиент согласен оплатить, это влияет только на количество требуемых последствий,
но само по себе не означает, что оператор выполнил блок мотивации.
""".strip()

In [81]:
hard_fail_rules = """
Критерий считается невыполненным, если:
- оператор не озвучил нужную мотивацию;
- оператор использовал запрещённую формулировку будущего последствия;
- оператор назвал точную дату наступления последствия;
- клиент относится к категории A, а оператор озвучил требование полного досрочного погашения или ст. 811 ГК РФ.

Если найдено одно из этих условий, criterion_met должен быть false, даже если клиент согласился оплатить.
""".strip()

In [82]:
def build_user_prompt(dialogue: str, overdue_days: int) -> str:
    return f"""
Оцени, насколько качественно оператор мотивирует клиента на оплату задолженности.

Входные данные:
Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории клиентов:
- {overdue_for_A}
- {overdue_for_B}

Как определить согласие клиента:
Согласен:
{payment_true}

Не согласен:
{payment_false}

Фокус оценки:
{evaluation_focus}

Правила оценки категории A:
{eval_rules_A}

Правила оценки категории Б:
{eval_rules_B}

Последствия категории A:
{consequences_A}

Дополнительные последствия категории Б:
{consequences_B}

Формы мотивации:
{motivation_forms}

Допустимые формулировки оператора:
{allowed_words}

Недопустимые формулировки оператора:
{not_allowed_words}

Исключения:
{exceptional_situations}

Правила автоматического зачёта:
{auto_pass_rules}

Правила ролей:
{role_rules}

Правила незачёта:
{fail_rules}

Правила обязательного незачёта:
{hard_fail_rules}

Общие ограничения:
{restrictions}

Порядок проверки:
{decision_steps}

Верни только валидный JSON без markdown-разметки и дополнительных комментариев.

Формат ответа:
{output_schema}
""".strip()

### Тест модификации

In [85]:
results_df, total_elapsed_time = run_evaluation(test_cases=test_cases,
                                                verbose=False,)

results_df

Evaluating dialogues:   0%|          | 0/9 [00:00<?, ?it/s]

,case_id,description,overdue_days,expected,predicted,is_correct,decision,client_category,client_status,exception_detected,interrupted_dialog,detected_consequences,violations,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",10,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'з...",[],21.01,Оператор дал позитивное уведомление об отмене ...
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",10,False,False,True,не выполнено,A,согласен оплатить,False,False,[],[],28.01,Оператор не озвучил ни одного последствия неоп...
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",10,False,False,True,незачёт,A,согласен оплатить,False,False,[],[Оператор не озвучил ни одного последствия нео...,45.07,"Клиент согласен оплатить, но оператор не озвуч..."
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",22,True,False,False,недостаточно данных,A,согласен оплатить,False,False,[],[],30.04,В диалоге не зафиксирована явная мотивация опе...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,15,False,False,True,недостаточно данных,A,согласен оплатить,False,False,[],[],12.08,В диалоге оператор не озвучил ни одного послед...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,18,True,False,False,недостаточно данных для автоматического зачёта,A,согласен оплатить,False,False,[],[],12.65,В диалоге оператор не попрощался в конце бесед...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,10,True,False,False,не выполнено,A,согласен оплатить,False,False,[],[],12.06,Оператор не озвучил ни одного обязательного по...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",22,True,False,False,не выполнено,A,соглаcен оплатить,False,False,[],[],12.29,"В диалоге клиент согласен оплатить, но операто..."
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,36,True,True,True,выполнено,Б,не согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'п...",[оператор назвал возможные последствия как мин...,25.50,К клиенту категории Б оператор упомянул послед...


In [86]:
summary_df = results_df[["case_id","description","expected","predicted","is_correct",
                         "decision","elapsed_time_sec","explanation",]]

summary_df

,case_id,description,expected,predicted,is_correct,decision,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",True,True,True,выполнено,21.01,Оператор дал позитивное уведомление об отмене ...
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",False,False,True,не выполнено,28.01,Оператор не озвучил ни одного последствия неоп...
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,False,True,незачёт,45.07,"Клиент согласен оплатить, но оператор не озвуч..."
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",True,False,False,недостаточно данных,30.04,В диалоге не зафиксирована явная мотивация опе...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,False,True,недостаточно данных,12.08,В диалоге оператор не озвучил ни одного послед...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,True,False,False,недостаточно данных для автоматического зачёта,12.65,В диалоге оператор не попрощался в конце бесед...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,True,False,False,не выполнено,12.06,Оператор не озвучил ни одного обязательного по...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",True,False,False,не выполнено,12.29,"В диалоге клиент согласен оплатить, но операто..."
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,True,True,True,выполнено,25.50,К клиенту категории Б оператор упомянул послед...


In [87]:
print("=== Итоги тестирования ===")
print(f"Всего кейсов: {len(results_df)}")
print(f"Корректных ответов: {results_df['is_correct'].sum()}")
print(f"Accuracy: {results_df['is_correct'].mean():.2%}")
print(f"Общее время: {total_elapsed_time:.2f} сек.")
print(f"Среднее время на кейс: {results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {results_df['elapsed_time_sec'].max():.2f} сек.")

=== Итоги тестирования ===
Всего кейсов: 9
Корректных ответов: 5
Accuracy: 55.56%
Общее время: 198.73 сек.
Среднее время на кейс: 22.08 сек.
Минимальное время: 12.06 сек.
Максимальное время: 45.07 сек.


In [88]:
incorrect_df = results_df[~results_df["is_correct"]]

incorrect_df[["case_id","description","expected","predicted","decision",
              "detected_consequences","violations","explanation",]]

,case_id,description,expected,predicted,decision,detected_consequences,violations,explanation
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",True,False,недостаточно данных,[],[],В диалоге не зафиксирована явная мотивация опе...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,True,False,недостаточно данных для автоматического зачёта,[],[],В диалоге оператор не попрощался в конце бесед...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,True,False,не выполнено,[],[],Оператор не озвучил ни одного обязательного по...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",True,False,не выполнено,[],[],"В диалоге клиент согласен оплатить, но операто..."


In [ ]:
for _, row in incorrect_df.iterrows():
    print("=" * 80)
    print("CASE:", row["case_id"])
    print("EXPECTED:", row["expected"])
    print("PREDICTED:", row["predicted"])
    print("\nEXPLANATION:")
    print(row["explanation"])
    print()

CASE: tz_02_A_agree_no_motivation
EXPECTED: False
PREDICTED: True

EXPLANATION:
Клиент выразил намерение оплатить, а оператор не назвал конкретных последствий неоплаты и не упоминал дату, что соответствует категории A при согласии клиента.

CASE: tz_03_A_wrong_period
EXPECTED: False
PREDICTED: True

EXPLANATION:
Категория A и статус клиента совпадают с согласие на оплату, однако оператор не озвучил ни одного последствия из категории A.

CASE: tz_05_A_forbidden_assertive_wording
EXPECTED: False
PREDICTED: True

EXPLANATION:
Срок просрочки 15 дней относится к категории A; оператор упомянул одно последствие неоплаты и сделал уведомление об передаче коллекторской службе, что соответствует требованиям категории A при согласии клиента.



СТАЛО ХУЖЕ, УПРОЩАЕМ

## Попытка убрать шум

In [89]:
SYSTEM_PROMPT = """
Ты оцениваешь, выполнен ли блок «Мотивация клиента на оплату» в диалоге оператора взыскания.
Верни только валидный JSON без markdown-разметки и дополнительных комментариев.
""".strip()

In [90]:
categories = """
Категория A: 1–30 дней просрочки.
Категория Б: 31–45 дней просрочки.
""".strip()

In [91]:
consequences = """
Последствия категории A:
- ухудшение кредитной истории;
- штрафы / неустойки;
- звонки;
- выезд сотрудника;
- продажа долга;
- передача в работу коллекторов.

Дополнительные последствия категории Б:
- требование полного досрочного погашения;
- ст. 811 ГК РФ;
- продажа долга;
- передача в работу коллекторов.
""".strip()

In [92]:
exceptions = """
Критерий считается выполненным автоматически, если:
- в диалоге упоминается банкротство, ЧС, военные действия, тюрьма, армия, мошенничество,
  смерть плательщика, инвалидность или лечение;
- оператор не попрощался в конце диалога.
""".strip()

In [93]:
evaluation_rules = """
Правила оценки:

1. Определи категорию клиента по сроку просрочки.

2. Определи, согласен ли клиент оплатить:
   - согласен: клиент подтверждает оплату или говорит, что оплатит;
   - не согласен: клиент отказывается, торгуется, просит отсрочку или не подтверждает оплату.

3. Для категории A:
   - если клиент согласен, нужна 1 мотивация;
   - если клиент не согласен, нужны 2 мотивации.

4. Для категории Б:
   - если клиент согласен, нужна 1 мотивация из категории A;
   - если клиент не согласен, нужна 1 мотивация из категории A и 1 дополнительная мотивация из категории Б.

5. Мотивацией считается:
   - последствие неоплаты;
   - констатация действующего последствия;
   - позитивное уведомление об отмене последствия после оплаты.

6. Недопустимо:
   - точные даты наступления последствий;
   - гарантии будущих последствий;
   - для категории A: требование полного досрочного погашения или ст. 811 ГК РФ.

7. Последствия и нарушения учитывай только из реплик оператора.
""".strip()

In [94]:
output_schema = """
{
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [],
  "violations": [],
  "explanation": "Краткое объяснение в 1 предложении."
}
""".strip()

In [95]:
def build_user_prompt(dialogue: str, overdue_days: int) -> str:
    return f"""
Оцени мотивацию оператора на оплату задолженности.

Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории:
{categories}

Последствия:
{consequences}

Исключения:
{exceptions}

Как оценивать:
{evaluation_rules}

Верни ответ строго в формате JSON:
{output_schema}
""".strip()

### Тест модификации

In [98]:
results_df, total_elapsed_time = run_evaluation(test_cases=test_cases,
                                                verbose=False,)

results_df

Evaluating dialogues:   0%|          | 0/9 [00:00<?, ?it/s]

,case_id,description,overdue_days,expected,predicted,is_correct,decision,client_category,client_status,exception_detected,interrupted_dialog,detected_consequences,violations,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",10,True,True,True,выполнено,A,согласен оплатить,False,False,[],[],12.29,Срок просрочки 10 дней относится к категории A...
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",10,False,False,True,не выполнено,A,согласен оплатить,False,False,[],[],6.27,Оператор не указал ни одного последствия неопл...
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",10,False,True,False,выполнено,A,согласен оплатить,False,False,[],[],27.77,По сроку просрочки 10 дней клиент относится к ...
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",22,True,True,True,выполнено,A,не согласен оплатить,False,False,[звонки],[],12.30,Срок просрочки 22 дня относится к категории A;...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,15,False,True,False,выполнено,A,не согласен оплатить,False,False,[],[],11.83,Категория A; оператор угрожает передать дело к...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,18,True,True,True,выполнено,A,согласен оплатить,True,False,[],[],60.07,Возрастная мотивация на оплату присутствует: к...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,10,True,True,True,выполнено,A,согласен оплатить,False,False,[уточнение: после оплаты обновим кредитную ист...,[],11.39,Срок просрочки 10 дней относится к категория A...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",22,True,True,True,выполнено,A,согласен оплатить,False,False,"[звонки, штрафы / неустойки, переплата, долг р...",[],12.94,Срок просрочки 22 дня (категория A); клиент вы...
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,36,True,True,True,выполнено,A,не согласен оплатить,False,False,[],[],12.54,Срок просрочки 36 дней относится к категории A...


In [99]:
summary_df = results_df[["case_id","description","expected","predicted","is_correct",
                         "decision","elapsed_time_sec","explanation",]]

summary_df

,case_id,description,expected,predicted,is_correct,decision,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",True,True,True,выполнено,12.29,Срок просрочки 10 дней относится к категории A...
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",False,False,True,не выполнено,6.27,Оператор не указал ни одного последствия неопл...
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,True,False,выполнено,27.77,По сроку просрочки 10 дней клиент относится к ...
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",True,True,True,выполнено,12.30,Срок просрочки 22 дня относится к категории A;...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,True,False,выполнено,11.83,Категория A; оператор угрожает передать дело к...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,True,True,True,выполнено,60.07,Возрастная мотивация на оплату присутствует: к...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,True,True,True,выполнено,11.39,Срок просрочки 10 дней относится к категория A...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",True,True,True,выполнено,12.94,Срок просрочки 22 дня (категория A); клиент вы...
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,True,True,True,выполнено,12.54,Срок просрочки 36 дней относится к категории A...


In [100]:
print("=== Итоги тестирования ===")
print(f"Всего кейсов: {len(results_df)}")
print(f"Корректных ответов: {results_df['is_correct'].sum()}")
print(f"Accuracy: {results_df['is_correct'].mean():.2%}")
print(f"Общее время: {total_elapsed_time:.2f} сек.")
print(f"Среднее время на кейс: {results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {results_df['elapsed_time_sec'].max():.2f} сек.")

=== Итоги тестирования ===
Всего кейсов: 9
Корректных ответов: 7
Accuracy: 77.78%
Общее время: 167.41 сек.
Среднее время на кейс: 18.60 сек.
Минимальное время: 6.27 сек.
Максимальное время: 60.07 сек.


In [101]:
incorrect_df = results_df[~results_df["is_correct"]]

incorrect_df[["case_id","description","expected","predicted","decision",
              "detected_consequences","violations","explanation",]]

,case_id,description,expected,predicted,decision,detected_consequences,violations,explanation
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,True,выполнено,[],[],По сроку просрочки 10 дней клиент относится к ...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,True,выполнено,[],[],Категория A; оператор угрожает передать дело к...


In [102]:
for _, row in incorrect_df.iterrows():
    print("=" * 80)
    print("CASE:", row["case_id"])
    print("EXPECTED:", row["expected"])
    print("PREDICTED:", row["predicted"])
    print("\nEXPLANATION:")
    print(row["explanation"])
    print()

CASE: tz_03_A_wrong_period
EXPECTED: False
PREDICTED: True

EXPLANATION:
По сроку просрочки 10 дней клиент относится к категории A; оператор не упоминает дополнительные мотивации при согласии клиента на оплату и в диалоге присутствует только сообщение о возможных последствиях, без упоминания конкретной даты или требований.

CASE: tz_05_A_forbidden_assertive_wording
EXPECTED: False
PREDICTED: True

EXPLANATION:
Категория A; оператор угрожает передать дело коллекторам и не прощается, клиент пока не выразил согласие на оплату.



## Финальный тест Baseline

In [104]:
realistic_test_cases = [
    {
        "case_id": "real_01_A_agree_positive_notification",
        "description": "Категория A, клиент согласен, оператор даёт позитивное уведомление",
        "overdue_days": 7,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченного платежа по договору.
Клиент: Да, я видел уведомление, сегодня внесу оплату.
Оператор: Правильно понимаю, что платёж будет сегодня?
Клиент: Да, сегодня вечером.
Оператор: После поступления платежа звонки по задолженности прекратятся.
Клиент: Хорошо, понял.
Оператор: Спасибо за информацию. До свидания.
""".strip(),
    },
    {
        "case_id": "real_02_A_agree_no_motivation",
        "description": "Категория A, клиент согласен, но оператор не мотивирует",
        "overdue_days": 11,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. Напоминаю о просроченной задолженности.
Клиент: Да, я в курсе, сегодня постараюсь оплатить.
Оператор: Хорошо, тогда ожидаем платёж. Пожалуйста, не затягивайте.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_03_A_disagree_two_A_consequences",
        "description": "Категория A, клиент не согласен, оператор называет два последствия A",
        "overdue_days": 23,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. У вас сохраняется просроченная задолженность. Когда сможете оплатить?
Клиент: Сейчас не могу, денег нет, возможно только через неделю.
Оператор: Понимаю. При отсутствии оплаты банк вправе начислять неустойку, из-за чего сумма долга может увеличиваться.
Оператор: Также могут продолжаться звонки вам и контактным лицам по вопросу задолженности.
Клиент: Понял, но раньше оплатить не получится.
Оператор: Какой срок оплаты можете назвать сейчас?
Клиент: Попробую решить вопрос до конца недели.
Оператор: Спасибо, информацию зафиксировал. До свидания.
""".strip(),
    },
    {
        "case_id": "real_04_A_wrong_period_full_repayment",
        "description": "Категория A, оператор использует последствие категории Б",
        "overdue_days": 19,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. По вашему договору есть просроченный платёж.
Клиент: Я понимаю, но сейчас оплатить не могу.
Оператор: В случае дальнейшей неоплаты банк может потребовать полного досрочного погашения всей задолженности.
Клиент: Но просрочка ведь меньше месяца.
Оператор: Я обязан предупредить о возможных последствиях.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_05_B_agree_A_consequence",
        "description": "Категория Б, клиент согласен, оператор называет последствие категории A",
        "overdue_days": 34,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность более месяца.
Клиент: Да, понимаю, завтра смогу внести платёж.
Оператор: Подтверждаете оплату завтра?
Клиент: Да, завтра оплачу.
Оператор: До поступления оплаты могут продолжаться звонки по задолженности.
Клиент: Хорошо, понял.
Оператор: Спасибо. До свидания.
""".strip(),
    },
    {
        "case_id": "real_06_B_agree_only_B_consequence",
        "description": "Категория Б, клиент согласен, но оператор называет только дополнительное последствие Б",
        "overdue_days": 39,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. У вас длительная просрочка по договору.
Клиент: Я смогу оплатить до пятницы.
Оператор: То есть оплату до пятницы подтверждаете?
Клиент: Да, подтверждаю.
Оператор: Банк может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_07_B_disagree_A_and_B_consequences",
        "description": "Категория Б, клиент не согласен, оператор называет последствие A и Б",
        "overdue_days": 41,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Пока не могу сказать, денег нет.
Оператор: В таком случае банк вправе начислять неустойку, и сумма задолженности может увеличиваться.
Оператор: Также возможно применение ст. 811 ГК РФ.
Клиент: Я понял, но сейчас всё равно оплатить не смогу.
Оператор: Как планируете решать вопрос?
Клиент: Буду искать деньги, но срок назвать не могу.
Оператор: Информацию зафиксировал. До свидания.
""".strip(),
    },
    {
        "case_id": "real_08_forbidden_exact_date_and_guarantee",
        "description": "Запрещённая формулировка: точная дата и гарантия последствия",
        "overdue_days": 16,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Я смогу оплатить только позже, сейчас денег нет.
Оператор: Если не оплатите до 20 мая, 21 мая мы передадим задолженность коллекторам.
Клиент: Я понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_09_exception_bankruptcy",
        "description": "Исключение: банкротство",
        "overdue_days": 27,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете внести оплату по задолженности?
Клиент: Сейчас не могу обсуждать оплату, я прохожу процедуру банкротства.
Оператор: Документы по процедуре уже поданы?
Клиент: Да, этим занимается юрист.
Оператор: Понял, информацию зафиксирую. До свидания.
""".strip(),
    },
    {
        "case_id": "real_10_interrupted_dialog_no_goodbye",
        "description": "Прерванный диалог: оператор не попрощался",
        "overdue_days": 21,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Пока не могу, возможно на следующей неделе.
Оператор: При отсутствии оплаты банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Оператор: Какой срок оплаты можете назвать?
""".strip(),
    },
]

In [105]:
realistic_results_df, realistic_total_time = run_evaluation(test_cases=realistic_test_cases,
                                                            verbose=False,)

realistic_results_df[["case_id","description","expected","predicted","is_correct",
                      "decision","detected_consequences","violations","explanation","elapsed_time_sec",]]

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

,case_id,description,expected,predicted,is_correct,decision,detected_consequences,violations,explanation,elapsed_time_sec
0,real_01_A_agree_positive_notification,"Категория A, клиент согласен, оператор даёт по...",True,True,True,выполнено,[],[],"Клиент выразил согласие оплатить сегодня, что ...",11.84
1,real_02_A_agree_no_motivation,"Категория A, клиент согласен, но оператор не м...",False,True,False,выполнено,[],[],Срок просрочки 11 дней относится к категории A...,27.79
2,real_03_A_disagree_two_A_consequences,"Категория A, клиент не согласен, оператор назы...",True,True,True,выполнено,[звонки],[],Категория A (1–30 дней просрочки); клиент выра...,12.19
3,real_04_A_wrong_period_full_repayment,"Категория A, оператор использует последствие к...",False,True,False,выполнено,[],[],Срок просрочки 19 дней относится к категории A...,27.23
4,real_05_B_agree_A_consequence,"Категория Б, клиент согласен, оператор называе...",True,True,True,выполнено,[],[],Срок просрочки 34 дня относится к категории Б;...,43.59
5,real_06_B_agree_only_B_consequence,"Категория Б, клиент согласен, но оператор назы...",False,True,False,выполнено,[требование полного досрочного погашения],[],Срок просрочки 39 дней относится к категории Б...,44.00
6,real_07_B_disagree_A_and_B_consequences,"Категория Б, клиент не согласен, оператор назы...",True,True,True,выполнено,[],[],Срок просрочки 41 день относится к категории Б...,28.59
7,real_08_forbidden_exact_date_and_guarantee,Запрещённая формулировка: точная дата и гарант...,False,True,False,выполнено,[],[],Срок просрочки 16 дней относится к категории A...,11.07
8,real_09_exception_bankruptcy,Исключение: банкротство,True,True,True,выполнено,"[звонки, передача в работу коллекторов]",[],Срок просрочки 27 дней — категория A; клиент н...,12.05
9,real_10_interrupted_dialog_no_goodbye,Прерванный диалог: оператор не попрощался,True,True,True,выполнено,"[звонки, выезд сотрудника, передача в работу к...",[],Срок просрочки 21 день соответствует категории...,12.18


In [106]:
print("=== Реалистичный контрольный тест ===")
print(f"Всего кейсов: {len(realistic_results_df)}")
print(f"Корректных ответов: {realistic_results_df['is_correct'].sum()}")
print(f"Accuracy: {realistic_results_df['is_correct'].mean():.2%}")
print(f"Общее время: {realistic_total_time:.2f} сек.")
print(f"Среднее время на кейс: {realistic_results_df['elapsed_time_sec'].mean():.2f} сек.")

=== Реалистичный контрольный тест ===
Всего кейсов: 10
Корректных ответов: 6
Accuracy: 60.00%
Общее время: 230.54 сек.
Среднее время на кейс: 23.05 сек.


In [107]:
realistic_incorrect_df = realistic_results_df[realistic_results_df["is_correct"] == False]

realistic_incorrect_df[["case_id","description","expected","predicted",
                        "detected_consequences","violations","explanation",]]

,case_id,description,expected,predicted,detected_consequences,violations,explanation
1,real_02_A_agree_no_motivation,"Категория A, клиент согласен, но оператор не м...",False,True,[],[],Срок просрочки 11 дней относится к категории A...
3,real_04_A_wrong_period_full_repayment,"Категория A, оператор использует последствие к...",False,True,[],[],Срок просрочки 19 дней относится к категории A...
5,real_06_B_agree_only_B_consequence,"Категория Б, клиент согласен, но оператор назы...",False,True,[требование полного досрочного погашения],[],Срок просрочки 39 дней относится к категории Б...
7,real_08_forbidden_exact_date_and_guarantee,Запрещённая формулировка: точная дата и гарант...,False,True,[],[],Срок просрочки 16 дней относится к категории A...


## Тестирование. Параметр 1 — наличие мотивации

### Примеры

In [ ]:
motivation_presence_test_cases = [
    {
        "case_id": "motivation_01_positive_notification_calls",
        "description": "Мотивация есть: позитивное уведомление о прекращении звонков после оплаты",
        "overdue_days": 10,
        "expected": True,
        "expected_has_motivation": True,
        "expected_motivation_type": "позитивное уведомление",
        "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченной задолженности.
Клиент: Да, я знаю, сегодня планирую оплатить.
Оператор: Подтверждаете, что платёж поступит сегодня до конца дня?
Клиент: Да, сегодня оплачу.
Оператор: Спасибо. Как только платёж поступит, звонки по задолженности прекратятся.
Клиент: Хорошо, понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_02_no_motivation_waiting_payment",
        "description": "Мотивации нет: оператор просто ждёт оплату",
        "overdue_days": 10,
        "expected": False,
        "expected_has_motivation": False,
        "expected_motivation_type": None,
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Да, я помню, сегодня постараюсь оплатить.
Оператор: Хорошо, ждём платёж. Себя не подводите.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_03_penalty_consequence",
        "description": "Мотивация есть: штрафы / неустойка",
        "overdue_days": 22,
        "expected": True,
        "expected_has_motivation": True,
        "expected_motivation_type": "штрафы / неустойки",
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете внести просроченный платёж?
Клиент: Сегодня не получится, денег пока нет.
Оператор: Понимаю. При этом банк вправе начислять неустойку, из-за чего сумма задолженности может увеличиваться.
Клиент: Да, понимаю.
Оператор: Как планируете решать вопрос с оплатой?
Клиент: Постараюсь найти деньги до конца недели.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_04_no_motivation_polite_reminder",
        "description": "Мотивации нет: вежливое напоминание без последствий",
        "overdue_days": 15,
        "expected": False,
        "expected_has_motivation": False,
        "expected_motivation_type": None,
        "dialogue": """
Оператор: Добрый день. Напоминаю, что по договору есть просроченный платёж.
Клиент: Я знаю, пока не успел оплатить.
Оператор: Когда ориентировочно сможете внести оплату?
Клиент: Думаю, завтра.
Оператор: Хорошо, тогда ожидаем поступление платежа.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_05_calls_consequence",
        "description": "Мотивация есть: звонки клиенту и контактным лицам",
        "overdue_days": 24,
        "expected": True,
        "expected_has_motivation": True,
        "expected_motivation_type": "звонки",
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность, нужно обсудить оплату.
Клиент: Пока не могу оплатить, зарплата только через неделю.
Оператор: В таком случае могут продолжаться звонки вам и контактным лицам по вопросу оплаты.
Клиент: Не хотелось бы, чтобы звонили родственникам.
Оператор: Тогда рекомендую как можно скорее определить дату платежа.
Клиент: Попробую внести часть завтра.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_06_no_motivation_payment_question_only",
        "description": "Мотивации нет: оператор только спрашивает дату оплаты",
        "overdue_days": 8,
        "expected": False,
        "expected_has_motivation": False,
        "expected_motivation_type": None,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Возможно, завтра.
Оператор: Завтра в первой половине дня или ближе к вечеру?
Клиент: Скорее вечером.
Оператор: Хорошо, информацию зафиксировал. Ожидаем оплату.
Клиент: Да, хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_07_credit_history_consequence",
        "description": "Мотивация есть: ухудшение кредитной истории",
        "overdue_days": 12,
        "expected": True,
        "expected_has_motivation": True,
        "expected_motivation_type": "ухудшение кредитной истории",
        "dialogue": """
Оператор: Добрый день. По вашему договору зафиксирована просроченная задолженность.
Клиент: Я понимаю, просто задержали оплату.
Оператор: Обращаю внимание, что просрочка может негативно повлиять на кредитную историю.
Клиент: Понял, постараюсь сегодня закрыть вопрос.
Оператор: Подтверждаете оплату сегодня?
Клиент: Да, сегодня.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_08_no_motivation_pressure_without_consequence",
        "description": "Мотивации нет: давление без конкретного последствия",
        "overdue_days": 18,
        "expected": False,
        "expected_has_motivation": False,
        "expected_motivation_type": None,
        "dialogue": """
Оператор: Добрый день. У вас задолженность, её нужно закрыть.
Клиент: Я понимаю, но сейчас не могу оплатить.
Оператор: Вы должны отнестись к этому серьёзно и не затягивать ситуацию.
Клиент: Я и так понимаю.
Оператор: Тогда ожидаем оплату при первой возможности.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_09_sale_of_debt_consequence",
        "description": "Мотивация есть: продажа долга",
        "overdue_days": 26,
        "expected": True,
        "expected_has_motivation": True,
        "expected_motivation_type": "продажа долга",
        "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченной задолженности.
Клиент: Сейчас не готов оплатить всю сумму.
Оператор: В случае дальнейшей неоплаты банк может продать долг третьим лицам.
Клиент: Я понял.
Оператор: Какой срок оплаты вы можете назвать?
Клиент: Попробую решить вопрос в течение недели.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "motivation_10_no_motivation_client_mentions_consequence",
        "description": "Мотивации нет: последствие озвучил клиент, а не оператор",
        "overdue_days": 14,
        "expected": False,
        "expected_has_motivation": False,
        "expected_motivation_type": None,
        "dialogue": """
Оператор: Добрый день. Когда сможете внести просроченный платёж?
Клиент: Я боюсь, что из-за просрочки испортится кредитная история.
Оператор: Понимаю ваше беспокойство. Когда сможете оплатить?
Клиент: Завтра.
Оператор: Хорошо, ожидаем поступление платежа.
Клиент: До свидания.
Оператор: До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
motivation_results_df, motivation_total_time = run_evaluation(
    test_cases=motivation_presence_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
def has_detected_consequences(value) -> bool:
    if value is None:
        return False

    if isinstance(value, list):
        return len(value) > 0

    return bool(value)

In [ ]:
motivation_results_df["expected_has_motivation"] = [
    case["expected_has_motivation"]
    for case in motivation_presence_test_cases
]

motivation_results_df["expected_motivation_type"] = [
    case["expected_motivation_type"]
    for case in motivation_presence_test_cases
]

motivation_results_df["predicted_has_motivation"] = motivation_results_df[
    "detected_consequences"
].apply(has_detected_consequences)

motivation_results_df["motivation_correct"] = (
    motivation_results_df["expected_has_motivation"]
    == motivation_results_df["predicted_has_motivation"]
)

In [ ]:
motivation_results_df[
    [
        "case_id",
        "description",
        "expected_has_motivation",
        "predicted_has_motivation",
        "motivation_correct",
        "expected_motivation_type",
        "detected_consequences",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,expected_has_motivation,predicted_has_motivation,motivation_correct,expected_motivation_type,detected_consequences,explanation,elapsed_time_sec
0,motivation_01_positive_notification_calls,Мотивация есть: позитивное уведомление о прекр...,True,True,True,позитивное уведомление,[{'text': 'звонки по задолженности прекратятся...,Клиент согласен оплатить. Оператор озвучил одн...,19.61
1,motivation_02_no_motivation_waiting_payment,Мотивации нет: оператор просто ждёт оплату,False,True,False,None,"[{'text': '', 'type': '', 'category': ''}]",Срок просрочки 10 дней попадает в категорию A....,11.34
2,motivation_03_penalty_consequence,Мотивация есть: штрафы / неустойка,True,True,True,штрафы / неустойки,[{'text': 'поскольку банк вправе начислять неу...,Срок просрочки 22 дня относится к категории A....,13.26
3,motivation_04_no_motivation_polite_reminder,Мотивации нет: вежливое напоминание без послед...,False,True,False,None,[{'text': 'около posibilidad позже оплаты прив...,Поскольку просрочка 15 дней подпадает под кате...,10.70
4,motivation_05_calls_consequence,Мотивация есть: звонки клиенту и контактным лицам,True,True,True,звонки,"[{'text': 'звонки', 'type': 'operational', 'ca...",Срок просрочки 24 дня. Клиент выразил намерени...,8.24
5,motivation_06_no_motivation_payment_question_only,Мотивации нет: оператор только спрашивает дату...,False,True,False,None,[{'text': 'волну необходимости оплаты в предло...,Клиент согласен оплатить. В диалоге оператор з...,8.31
6,motivation_07_credit_history_consequence,Мотивация есть: ухудшение кредитной истории,True,True,True,ухудшение кредитной истории,[{'text': 'просрочка может негативно повлиять ...,Срок просрочки 12 дней попадает в категорию A ...,41.27
7,motivation_08_no_motivation_pressure_without_c...,Мотивации нет: давление без конкретного послед...,False,True,False,None,"[{'text': 'передача в работу коллекторов', 'ty...",Срок просрочки 18 дней относится к категории A...,24.37
8,motivation_09_sale_of_debt_consequence,Мотивация есть: продажа долга,True,True,True,продажа долга,"[{'text': 'продажа долга', 'type': 'потенциаль...",Срок просрочки 26 дней относится к категории A...,8.55
9,motivation_10_no_motivation_client_mentions_co...,"Мотивации нет: последствие озвучил клиент, а н...",False,True,False,None,"[{'text': 'звонки', 'type': 'последствие', 'ca...",Срок просрочки 14 дней относится к категории A...,24.85


In [ ]:
print("=== Проверка наличия мотивации ===")
print(f"Всего кейсов: {len(motivation_results_df)}")
print(f"Корректных ответов: {motivation_results_df['motivation_correct'].sum()}")
print(f"Accuracy по наличию мотивации: {motivation_results_df['motivation_correct'].mean():.2%}")
print(f"Общее время: {motivation_total_time:.2f} сек.")
print(f"Среднее время на кейс: {motivation_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {motivation_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {motivation_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка наличия мотивации ===
Всего кейсов: 10
Корректных ответов: 5
Accuracy по наличию мотивации: 50.00%
Общее время: 170.52 сек.
Среднее время на кейс: 17.05 сек.
Минимальное время: 8.24 сек.
Максимальное время: 41.27 сек.


In [ ]:
motivation_incorrect_df = motivation_results_df[
    motivation_results_df["motivation_correct"] == False
]

motivation_incorrect_df[
    [
        "case_id",
        "description",
        "expected_has_motivation",
        "predicted_has_motivation",
        "expected_motivation_type",
        "detected_consequences",
        "violations",
        "explanation",
    ]
]

,case_id,description,expected_has_motivation,predicted_has_motivation,expected_motivation_type,detected_consequences,violations,explanation
1,motivation_02_no_motivation_waiting_payment,Мотивации нет: оператор просто ждёт оплату,False,True,None,"[{'text': '', 'type': '', 'category': ''}]",[],Срок просрочки 10 дней попадает в категорию A....
3,motivation_04_no_motivation_polite_reminder,Мотивации нет: вежливое напоминание без послед...,False,True,None,[{'text': 'около posibilidad позже оплаты прив...,[],Поскольку просрочка 15 дней подпадает под кате...
5,motivation_06_no_motivation_payment_question_only,Мотивации нет: оператор только спрашивает дату...,False,True,None,[{'text': 'волну необходимости оплаты в предло...,[],Клиент согласен оплатить. В диалоге оператор з...
7,motivation_08_no_motivation_pressure_without_c...,Мотивации нет: давление без конкретного послед...,False,True,None,"[{'text': 'передача в работу коллекторов', 'ty...",[],Срок просрочки 18 дней относится к категории A...
9,motivation_10_no_motivation_client_mentions_co...,"Мотивации нет: последствие озвучил клиент, а н...",False,True,None,"[{'text': 'звонки', 'type': 'последствие', 'ca...",[],Срок просрочки 14 дней относится к категории A...


## Тестирование. Параметр 2 — согласился ли клиент на оплату задолженности

### Примеры

In [ ]:
client_agreement_test_cases = [
    {
        "case_id": "agreement_01_explicit_today_payment",
        "description": "Клиент явно соглашается оплатить сегодня",
        "overdue_days": 10,
        "expected": True,
        "expected_client_status": "согласен оплатить",
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность, нужно обсудить оплату.
Клиент: Да, я знаю.
Оператор: Сможете внести платёж сегодня до конца дня?
Клиент: Да, сегодня оплачу.
Оператор: Спасибо. После поступления оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_02_direct_refusal",
        "description": "Клиент прямо отказывается платить сейчас",
        "overdue_days": 12,
        "expected": False,
        "expected_client_status": "не согласен оплатить",
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Сейчас платить не буду, у меня нет денег.
Оператор: Правильно понимаю, что оплату сегодня вы не подтверждаете?
Клиент: Да, не подтверждаю.
Оператор: В таком случае банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Клиент: Понимаю.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_03_asks_for_delay",
        "description": "Клиент просит отсрочку",
        "overdue_days": 15,
        "expected": False,
        "expected_client_status": "не согласен оплатить",
        "dialogue": """
Оператор: Добрый день. Просроченный платёж необходимо внести как можно скорее.
Клиент: Я понимаю, но прошу подождать ещё неделю.
Оператор: То есть в предложенный срок оплату внести не сможете?
Клиент: Да, пока не смогу.
Оператор: Банк может начислять штрафы и неустойку при дальнейшем отсутствии оплаты.
Клиент: Хорошо, понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_04_ugu_confirmation",
        "description": "Клиент подтверждает оплату коротким ответом «угу»",
        "overdue_days": 10,
        "expected": True,
        "expected_client_status": "согласен оплатить",
        "dialogue": """
Оператор: Добрый день. Подтверждаете оплату до пятницы?
Клиент: Угу.
Оператор: Отлично. После оплаты информация по кредитной истории обновится.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_05_maybe_try_not_confirmed",
        "description": "Клиент говорит «постараюсь», но не подтверждает оплату",
        "overdue_days": 18,
        "expected": False,
        "expected_client_status": "не согласен оплатить",
        "dialogue": """
Оператор: Добрый день. Сможете внести платёж сегодня?
Клиент: Не знаю, постараюсь, если получится.
Оператор: То есть точно оплату сегодня вы не подтверждаете?
Клиент: Нет, точно обещать не могу.
Оператор: При отсутствии оплаты банк вправе начислять неустойку.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_06_partial_payment_only",
        "description": "Клиент готов внести только часть суммы",
        "overdue_days": 22,
        "expected": False,
        "expected_client_status": "не согласен оплатить",
        "dialogue": """
Оператор: Добрый день. Сегодня необходимо закрыть просроченную задолженность полностью.
Клиент: Полностью не смогу, могу внести только небольшую часть.
Оператор: Полную оплату сегодня подтверждаете?
Клиент: Нет, всю сумму сегодня не внесу.
Оператор: Банк вправе начислять штрафы и неустойку, из-за чего долг может увеличиваться.
Клиент: Понимаю.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_07_specific_date_confirmed",
        "description": "Клиент подтверждает оплату в конкретный предложенный срок",
        "overdue_days": 14,
        "expected": True,
        "expected_client_status": "согласен оплатить",
        "dialogue": """
Оператор: Добрый день. Можете оплатить задолженность до пятницы включительно?
Клиент: Да, до пятницы внесу оплату.
Оператор: Подтверждаете этот срок?
Клиент: Да, подтверждаю.
Оператор: После поступления платежа звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_08_negotiation_then_agreement",
        "description": "Клиент сначала спорит, затем подтверждает оплату",
        "overdue_days": 25,
        "expected": True,
        "expected_client_status": "согласен оплатить",
        "dialogue": """
Оператор: Добрый день. Когда сможете внести просроченный платёж?
Клиент: Сегодня не хотелось бы, у меня другие расходы.
Оператор: Понимаю. Но при отсутствии оплаты банк может продолжить начисление неустойки.
Клиент: Тогда хорошо, внесу платёж завтра утром.
Оператор: Правильно понимаю, завтра утром оплату подтверждаете?
Клиент: Да, подтверждаю.
Оператор: Спасибо. До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_09_no_promise",
        "description": "Клиент не обещает оплату",
        "overdue_days": 30,
        "expected": False,
        "expected_client_status": "не согласен оплатить",
        "dialogue": """
Оператор: Добрый день. Когда сможете закрыть просроченную задолженность?
Клиент: Пока не знаю, ничего обещать не могу.
Оператор: Можете назвать хотя бы ориентировочную дату оплаты?
Клиент: Нет, не могу.
Оператор: При дальнейшем отсутствии оплаты могут продолжаться звонки по вопросу задолженности.
Клиент: Понимаю.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "agreement_10_client_mentions_future_without_commitment",
        "description": "Клиент говорит о будущем платеже, но без подтверждения",
        "overdue_days": 35,
        "expected": False,
        "expected_client_status": "не согласен оплатить",
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность. Когда сможете оплатить?
Клиент: Наверное, когда получу деньги, тогда и внесу.
Оператор: На этой неделе оплату подтверждаете?
Клиент: Нет, на этой неделе не подтверждаю.
Оператор: В таком случае банк вправе начислять неустойку и может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
agreement_results_df, agreement_total_time = run_evaluation(
    test_cases=client_agreement_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
agreement_results_df["expected_client_status"] = [
    case["expected_client_status"]
    for case in client_agreement_test_cases
]

In [ ]:
def normalize_client_status(status: str) -> str:
    if not isinstance(status, str):
        return "unknown"

    status = status.lower().strip()

    if "не соглас" in status:
        return "не согласен оплатить"

    if "соглас" in status:
        return "согласен оплатить"

    return "unknown"

In [ ]:
agreement_results_df["predicted_client_status_norm"] = agreement_results_df[
    "client_status"
].apply(normalize_client_status)

agreement_results_df["client_status_correct"] = (
    agreement_results_df["expected_client_status"]
    == agreement_results_df["predicted_client_status_norm"]
)

In [ ]:
agreement_results_df[
    [
        "case_id",
        "description",
        "expected_client_status",
        "client_status",
        "predicted_client_status_norm",
        "client_status_correct",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,expected_client_status,client_status,predicted_client_status_norm,client_status_correct,explanation,elapsed_time_sec
0,agreement_01_explicit_today_payment,Клиент явно соглашается оплатить сегодня,согласен оплатить,согласен оплатить,согласен оплатить,True,Клиент согласился оплатить в предложенный опер...,23.92
1,agreement_02_direct_refusal,Клиент прямо отказывается платить сейчас,не согласен оплатить,не согласен оплатить,не согласен оплатить,True,По диалогу срок просрочки 12 дней попадает в к...,8.83
2,agreement_03_asks_for_delay,Клиент просит отсрочку,не согласен оплатить,не согласен,не согласен оплатить,True,"Клиент просрочил 15 дней → категория A, не сог...",25.66
3,agreement_04_ugu_confirmation,Клиент подтверждает оплату коротким ответом «угу»,согласен оплатить,согласен оплатить,согласен оплатить,True,Клиент выразил согласие оплатить (угу) и опера...,22.08
4,agreement_05_maybe_try_not_confirmed,"Клиент говорит «постараюсь», но не подтверждае...",не согласен оплатить,не согласен оплатить,не согласен оплатить,True,Клиент не согласен оплатить (показано как «Нет...,8.52
5,agreement_06_partial_payment_only,Клиент готов внести только часть суммы,не согласен оплатить,согласен оплатить,согласен оплатить,False,Срок просрочки 22 дня относится к категории A....,10.32
6,agreement_07_specific_date_confirmed,Клиент подтверждает оплату в конкретный предло...,согласен оплатить,согласен оплатить,согласен оплатить,True,Срок просрочки 14 дней относится к категории A...,7.30
7,agreement_08_negotiation_then_agreement,"Клиент сначала спорит, затем подтверждает оплату",согласен оплатить,согласен оплатить,согласен оплатить,True,Согласно диалогу клиент соглашается внести пла...,7.70
8,agreement_09_no_promise,Клиент не обещает оплату,не согласен оплатить,нет явного согласия на оплату,согласен оплатить,False,Срок просрочки 30 дней относится к категории A...,7.85
9,agreement_10_client_mentions_future_without_co...,"Клиент говорит о будущем платеже, но без подтв...",не согласен оплатить,не согласен оплатить,не согласен оплатить,True,Срок просрочки 35 дней соответствует категории...,24.29


In [ ]:
print("=== Проверка статуса клиента ===")
print(f"Всего кейсов: {len(agreement_results_df)}")
print(f"Корректных ответов: {agreement_results_df['client_status_correct'].sum()}")
print(f"Accuracy по статусу клиента: {agreement_results_df['client_status_correct'].mean():.2%}")
print(f"Общее время: {agreement_total_time:.2f} сек.")
print(f"Среднее время на кейс: {agreement_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {agreement_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {agreement_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка статуса клиента ===
Всего кейсов: 10
Корректных ответов: 8
Accuracy по статусу клиента: 80.00%
Общее время: 146.51 сек.
Среднее время на кейс: 14.65 сек.
Минимальное время: 7.30 сек.
Максимальное время: 25.66 сек.


In [ ]:
agreement_incorrect_df = agreement_results_df[
    agreement_results_df["client_status_correct"] == False
]

agreement_incorrect_df[
    [
        "case_id",
        "description",
        "expected_client_status",
        "client_status",
        "predicted_client_status_norm",
        "violations",
        "explanation",
    ]
]

,case_id,description,expected_client_status,client_status,predicted_client_status_norm,violations,explanation
5,agreement_06_partial_payment_only,Клиент готов внести только часть суммы,не согласен оплатить,согласен оплатить,согласен оплатить,[],Срок просрочки 22 дня относится к категории A....
8,agreement_09_no_promise,Клиент не обещает оплату,не согласен оплатить,нет явного согласия на оплату,согласен оплатить,[],Срок просрочки 30 дней относится к категории A...


## Тестирование. Параметр 3 — верно ли определена категория клиента

### Примеры

In [ ]:
category_detection_test_cases = [
    {
        "case_id": "category_01_A_lower_boundary",
        "description": "Категория A: нижняя граница, 1 день просрочки",
        "overdue_days": 1,
        "expected": True,
        "expected_category": "A",
        "dialogue": """
Оператор: Добрый день. У вас первый день просроченной задолженности.
Клиент: Да, я знаю, сегодня оплачу.
Оператор: После поступления оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_02_A_middle",
        "description": "Категория A: середина диапазона",
        "overdue_days": 15,
        "expected": True,
        "expected_category": "A",
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Сегодня оплатить не смогу.
Оператор: Банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Оператор: Также могут продолжаться звонки по вопросу оплаты.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_03_A_upper_boundary",
        "description": "Категория A: верхняя граница, 30 дней просрочки",
        "overdue_days": 30,
        "expected": True,
        "expected_category": "A",
        "dialogue": """
Оператор: Добрый день. По договору есть просроченная задолженность.
Клиент: Пока не могу назвать точную дату оплаты.
Оператор: Банк может продолжить начисление штрафов и неустойки.
Оператор: Также могут продолжаться звонки вам и контактным лицам.
Клиент: Понимаю.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_04_B_lower_boundary",
        "description": "Категория Б: нижняя граница, 31 день просрочки",
        "overdue_days": 31,
        "expected": True,
        "expected_category": "Б",
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность.
Клиент: Сейчас оплатить не могу.
Оператор: Банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Оператор: Также банк может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_05_B_middle",
        "description": "Категория Б: середина диапазона",
        "overdue_days": 38,
        "expected": True,
        "expected_category": "Б",
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату?
Клиент: Пока не могу сказать.
Оператор: При дальнейшем отсутствии оплаты банк вправе начислять неустойку.
Оператор: Также возможно применение ст. 811 ГК РФ.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_06_B_upper_boundary",
        "description": "Категория Б: верхняя граница, 45 дней просрочки",
        "overdue_days": 45,
        "expected": True,
        "expected_category": "Б",
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность по кредитному договору.
Клиент: Да, понимаю, но сейчас оплатить не могу.
Оператор: Банк вправе начислять неустойку.
Оператор: Также банк может потребовать полного досрочного погашения всей задолженности.
Клиент: Я понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_07_A_wrong_period_consequence",
        "description": "Категория A: 10 дней, но оператор озвучивает последствие категории Б",
        "overdue_days": 10,
        "expected": False,
        "expected_category": "A",
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность.
Клиент: Сегодня оплатить не смогу.
Оператор: Банк может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_08_B_not_enough_consequences",
        "description": "Категория Б: 35 дней, но озвучено только последствие категории A",
        "overdue_days": 35,
        "expected": False,
        "expected_category": "Б",
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Пока не знаю, денег нет.
Оператор: Банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_09_A_dialogue_says_long_delay",
        "description": "Категория A: 29 дней, несмотря на формулировку оператора про длительную просрочку",
        "overdue_days": 29,
        "expected": True,
        "expected_category": "A",
        "dialogue": """
Оператор: Добрый день. У вас уже достаточно длительная просроченная задолженность.
Клиент: Да, понимаю, но пока не могу оплатить.
Оператор: Банк вправе начислять штрафы и неустойку.
Оператор: Также могут продолжаться звонки по вопросу оплаты.
Клиент: Хорошо, я постараюсь решить вопрос.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "category_10_B_client_promises_payment",
        "description": "Категория Б: 32 дня, клиент согласен оплатить",
        "overdue_days": 32,
        "expected": True,
        "expected_category": "Б",
        "dialogue": """
Оператор: Добрый день. Подтверждаете оплату до пятницы?
Клиент: Да, до пятницы внесу оплату.
Оператор: После поступления платежа звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
category_results_df, category_total_time = run_evaluation(
    test_cases=category_detection_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
category_results_df["expected_category"] = [
    case["expected_category"]
    for case in category_detection_test_cases
]

In [ ]:
def normalize_client_category(category: str) -> str:
    if not isinstance(category, str):
        return "unknown"

    category = category.strip().upper()

    if category in ["A", "А"]:
        return "A"

    if category in ["Б", "B"]:
        return "Б"

    return "unknown"

In [ ]:
category_results_df["predicted_category_norm"] = category_results_df[
    "client_category"
].apply(normalize_client_category)

category_results_df["category_correct"] = (
    category_results_df["expected_category"]
    == category_results_df["predicted_category_norm"]
)

In [ ]:
category_results_df[
    [
        "case_id",
        "description",
        "overdue_days",
        "expected_category",
        "client_category",
        "predicted_category_norm",
        "category_correct",
        "expected",
        "predicted",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,overdue_days,expected_category,client_category,predicted_category_norm,category_correct,expected,predicted,explanation,elapsed_time_sec
0,category_01_A_lower_boundary,"Категория A: нижняя граница, 1 день просрочки",1,A,A,A,True,True,True,Клиент в рамках категории A согласен оплатить....,6.17
1,category_02_A_middle,Категория A: середина диапазона,15,A,A,A,True,True,True,Клиент выразил согласие оплатить. В репликах о...,37.66
2,category_03_A_upper_boundary,"Категория A: верхняя граница, 30 дней просрочки",30,A,A,A,True,True,True,Срок просрочки 30 дней относится к категории A...,6.62
3,category_04_B_lower_boundary,"Категория Б: нижняя граница, 31 день просрочки",31,Б,Б,Б,True,True,True,Диалог содержит заявление оператора о потенциа...,22.08
4,category_05_B_middle,Категория Б: середина диапазона,38,Б,Б,Б,True,True,True,Оператор не уточнил последствия в явном виде д...,7.30
5,category_06_B_upper_boundary,"Категория Б: верхняя граница, 45 дней просрочки",45,Б,Б,Б,True,True,True,Оператор упомянул последствия категории Б: тре...,7.09
6,category_07_A_wrong_period_consequence,"Категория A: 10 дней, но оператор озвучивает п...",10,A,A,A,True,False,False,"Клиент не согласен оплатить, поэтому оператор ...",0.58
7,category_08_B_not_enough_consequences,"Категория Б: 35 дней, но озвучено только после...",35,Б,Б,Б,True,False,True,Диалог содержит указания оператором последстви...,23.98
8,category_09_A_dialogue_says_long_delay,"Категория A: 29 дней, несмотря на формулировку...",29,A,A,A,True,True,True,Оператор сообщил о возможной необходимости опл...,9.55
9,category_10_B_client_promises_payment,"Категория Б: 32 дня, клиент согласен оплатить",32,Б,Б,Б,True,True,True,,8.06


In [ ]:
print("=== Проверка определения категории клиента ===")
print(f"Всего кейсов: {len(category_results_df)}")
print(f"Корректных категорий: {category_results_df['category_correct'].sum()}")
print(f"Accuracy по категории: {category_results_df['category_correct'].mean():.2%}")
print(f"Общее время: {category_total_time:.2f} сек.")
print(f"Среднее время на кейс: {category_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {category_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {category_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка определения категории клиента ===
Всего кейсов: 10
Корректных категорий: 10
Accuracy по категории: 100.00%
Общее время: 129.12 сек.
Среднее время на кейс: 12.91 сек.
Минимальное время: 0.58 сек.
Максимальное время: 37.66 сек.


In [ ]:
category_incorrect_df = category_results_df[
    category_results_df["category_correct"] == False
]

category_incorrect_df[
    [
        "case_id",
        "description",
        "overdue_days",
        "expected_category",
        "client_category",
        "predicted_category_norm",
        "violations",
        "explanation",
    ]
]

,case_id,description,overdue_days,expected_category,client_category,predicted_category_norm,violations,explanation


## Тестирование. Параметр 4 — верно ли выбрана мотивация относительно категории

### Примеры

In [ ]:
category_motivation_test_cases = [
    {
        "case_id": "catmot_01_A_agree_one_A_consequence",
        "description": "Категория A, клиент согласен: одного последствия категории A достаточно",
        "overdue_days": 10,
        "expected": True,
        "expected_category": "A",
        "expected_client_status": "согласен оплатить",
        "expected_category_motivation_valid": True,
        "expected_required_pattern": "A + согласен: 1 последствие категории A",
        "expected_consequence_types": ["звонки"],
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Да, я знаю.
Оператор: Сможете оплатить сегодня?
Клиент: Да, сегодня внесу оплату.
Оператор: После поступления оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_02_A_agree_no_consequence",
        "description": "Категория A, клиент согласен: последствие не озвучено",
        "overdue_days": 12,
        "expected": False,
        "expected_category": "A",
        "expected_client_status": "согласен оплатить",
        "expected_category_motivation_valid": False,
        "expected_required_pattern": "A + согласен: нужно 1 последствие категории A",
        "expected_consequence_types": [],
        "dialogue": """
Оператор: Добрый день. У вас просроченный платёж.
Клиент: Да, сегодня оплачу.
Оператор: Хорошо, ждём платёж. Пожалуйста, не затягивайте.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_03_A_disagree_two_A_consequences",
        "description": "Категория A, клиент не согласен: два последствия категории A",
        "overdue_days": 22,
        "expected": True,
        "expected_category": "A",
        "expected_client_status": "не согласен оплатить",
        "expected_category_motivation_valid": True,
        "expected_required_pattern": "A + не согласен: 2 последствия категории A",
        "expected_consequence_types": ["штрафы / неустойки", "звонки"],
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Сегодня не смогу, денег нет.
Оператор: В таком случае банк вправе начислять неустойку, из-за чего сумма задолженности может увеличиваться.
Оператор: Также могут продолжаться звонки вам и контактным лицам по вопросу оплаты.
Клиент: Понимаю.
Оператор: Как планируете решать вопрос?
Клиент: Постараюсь найти деньги до конца недели.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_04_A_disagree_one_A_consequence",
        "description": "Категория A, клиент не согласен: озвучено только одно последствие категории A",
        "overdue_days": 24,
        "expected": False,
        "expected_category": "A",
        "expected_client_status": "не согласен оплатить",
        "expected_category_motivation_valid": False,
        "expected_required_pattern": "A + не согласен: нужно 2 последствия категории A",
        "expected_consequence_types": ["штрафы / неустойки"],
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату?
Клиент: Сейчас не могу, возможно, через неделю.
Оператор: При отсутствии оплаты банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Клиент: Понял, но быстрее не получится.
Оператор: Тогда ожидаем оплату при первой возможности.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_05_B_agree_one_A_consequence",
        "description": "Категория Б, клиент согласен: одного последствия категории A достаточно",
        "overdue_days": 35,
        "expected": True,
        "expected_category": "Б",
        "expected_client_status": "согласен оплатить",
        "expected_category_motivation_valid": True,
        "expected_required_pattern": "Б + согласен: 1 последствие категории A",
        "expected_consequence_types": ["звонки"],
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность.
Клиент: Да, понимаю.
Оператор: Подтверждаете оплату до пятницы?
Клиент: Да, до пятницы внесу платёж.
Оператор: После поступления оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_06_B_agree_only_B_consequence",
        "description": "Категория Б, клиент согласен: озвучено только последствие категории Б",
        "overdue_days": 36,
        "expected": False,
        "expected_category": "Б",
        "expected_client_status": "согласен оплатить",
        "expected_category_motivation_valid": False,
        "expected_required_pattern": "Б + согласен: нужно 1 последствие категории A",
        "expected_consequence_types": ["требование полного досрочного погашения"],
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность более месяца.
Клиент: Я смогу оплатить до пятницы.
Оператор: Подтверждаете оплату до пятницы?
Клиент: Да, подтверждаю.
Оператор: Банк может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_07_B_disagree_A_and_B_consequences",
        "description": "Категория Б, клиент не согласен: есть последствие A и последствие Б",
        "overdue_days": 40,
        "expected": True,
        "expected_category": "Б",
        "expected_client_status": "не согласен оплатить",
        "expected_category_motivation_valid": True,
        "expected_required_pattern": "Б + не согласен: 1 последствие A + 1 последствие Б",
        "expected_consequence_types": ["штрафы / неустойки", "требование полного досрочного погашения"],
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Пока не могу назвать срок, денег нет.
Оператор: В таком случае банк вправе начислять неустойку, и сумма задолженности может увеличиваться.
Оператор: Также банк может потребовать полного досрочного погашения всей задолженности.
Клиент: Я понял, но сейчас оплатить не могу.
Оператор: Как планируете решать вопрос?
Клиент: Буду искать деньги.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_08_B_disagree_only_A_consequence",
        "description": "Категория Б, клиент не согласен: есть только последствие A",
        "overdue_days": 42,
        "expected": False,
        "expected_category": "Б",
        "expected_client_status": "не согласен оплатить",
        "expected_category_motivation_valid": False,
        "expected_required_pattern": "Б + не согласен: не хватает последствия категории Б",
        "expected_consequence_types": ["штрафы / неустойки"],
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность.
Клиент: Да, но сейчас оплатить не могу.
Оператор: Банк вправе начислять неустойку, из-за чего сумма задолженности может увеличиваться.
Клиент: Понимаю.
Оператор: Когда сможете внести оплату?
Клиент: Пока не знаю.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_09_B_disagree_only_B_consequence",
        "description": "Категория Б, клиент не согласен: есть только последствие Б",
        "overdue_days": 38,
        "expected": False,
        "expected_category": "Б",
        "expected_client_status": "не согласен оплатить",
        "expected_category_motivation_valid": False,
        "expected_required_pattern": "Б + не согласен: не хватает последствия категории A",
        "expected_consequence_types": ["ст. 811 ГК РФ"],
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату?
Клиент: Не знаю, пока возможности нет.
Оператор: В таком случае возможно применение ст. 811 ГК РФ.
Клиент: Понял.
Оператор: Какой срок оплаты можете назвать?
Клиент: Пока никакой.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "catmot_10_A_agree_B_only_period_error",
        "description": "Категория A, клиент согласен: озвучено только последствие категории Б",
        "overdue_days": 14,
        "expected": False,
        "expected_category": "A",
        "expected_client_status": "согласен оплатить",
        "expected_category_motivation_valid": False,
        "expected_required_pattern": "A + согласен: последствие категории Б не подходит для A",
        "expected_consequence_types": ["требование полного досрочного погашения"],
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Я могу оплатить сегодня вечером.
Оператор: Подтверждаете оплату сегодня?
Клиент: Да, подтверждаю.
Оператор: Банк может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
category_motivation_results_df, category_motivation_total_time = run_evaluation(
    test_cases=category_motivation_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
category_motivation_results_df["expected_category"] = [case["expected_category"]
    for case in category_motivation_test_cases]

category_motivation_results_df["expected_client_status"] = [case["expected_client_status"]
    for case in category_motivation_test_cases]

category_motivation_results_df["expected_category_motivation_valid"] = [case["expected_category_motivation_valid"]
    for case in category_motivation_test_cases]

category_motivation_results_df["expected_required_pattern"] = [case["expected_required_pattern"]
    for case in category_motivation_test_cases]

category_motivation_results_df["expected_consequence_types"] = [case["expected_consequence_types"]
    for case in category_motivation_test_cases]

In [ ]:
category_motivation_results_df["category_motivation_correct"] = (
category_motivation_results_df["predicted"] == category_motivation_results_df["expected_category_motivation_valid"])

In [ ]:
category_motivation_results_df[["case_id","description","expected_category",
                                "client_category","expected_client_status",
                                "client_status","expected_required_pattern",
                                "expected_category_motivation_valid","predicted",
                                "category_motivation_correct","expected_consequence_types",
                                "detected_consequences","violations","explanation",
                                "elapsed_time_sec",]]

,case_id,description,expected_category,client_category,expected_client_status,client_status,expected_required_pattern,expected_category_motivation_valid,predicted,category_motivation_correct,expected_consequence_types,detected_consequences,violations,explanation,elapsed_time_sec
0,catmot_01_A_agree_one_A_consequence,"Категория A, клиент согласен: одного последств...",A,A,согласен оплатить,согласен оплатить,A + согласен: 1 последствие категории A,True,True,True,[звонки],"[{'text': 'последствия неоплаты: звонки', 'typ...",[],Клиент согласен оплатить. В реплике оператора ...,45.15
1,catmot_02_A_agree_no_consequence,"Категория A, клиент согласен: последствие не о...",A,A,согласен оплатить,согласен оплатить,A + согласен: нужно 1 последствие категории A,False,True,False,[],"[{'text': 'пожалуйста, не затягивайте', 'type'...",[],Срок просрочки 12 дней относится к категории A...,12.96
2,catmot_03_A_disagree_two_A_consequences,"Категория A, клиент не согласен: два последств...",A,A,не согласен оплатить,не согласен,A + не согласен: 2 последствия категории A,True,True,True,"[штрафы / неустойки, звонки]","[{'text': 'банк вправе начислять неустойку', '...",[],Оператор озвучил два последствия категории A (...,21.90
3,catmot_04_A_disagree_one_A_consequence,"Категория A, клиент не согласен: озвучено толь...",A,A,не согласен оплатить,согласен оплатить,A + не согласен: нужно 2 последствия категории A,False,True,False,[штрафы / неустойки],"[{'text': 'банк вправе начислять неустойку, су...",[],Срок просрочки 24 дня относится к категории A....,28.39
4,catmot_05_B_agree_one_A_consequence,"Категория Б, клиент согласен: одного последств...",Б,Б,согласен оплатить,согласен оплатить,Б + согласен: 1 последствие категории A,True,True,True,[звонки],"[{'text': '', 'type': '', 'category': ''}]",[],Срок просрочки 35 дней относится к категории Б...,12.32
5,catmot_06_B_agree_only_B_consequence,"Категория Б, клиент согласен: озвучено только ...",Б,Б,согласен оплатить,согласен оплатить,Б + согласен: нужно 1 последствие категории A,False,True,False,[требование полного досрочного погашения],"[{'text': 'передача в работу коллекторов', 'ty...",[],Срок просрочки 36 дней относится к категории Б...,28.14
6,catmot_07_B_disagree_A_and_B_consequences,"Категория Б, клиент не согласен: есть последст...",Б,Б,не согласен оплатить,не согласен оплатить,Б + не согласен: 1 последствие A + 1 последств...,True,True,True,"[штрафы / неустойки, требование полного досроч...",[{'text': 'предъявлено требование полного доср...,[],Срок просрочки 40 дней относится к категории Б...,14.70
7,catmot_08_B_disagree_only_A_consequence,"Категория Б, клиент не согласен: есть только п...",Б,Б,не согласен оплатить,не согласен оплатить,Б + не согласен: не хватает последствия катего...,False,True,False,[штрафы / неустойки],"[{'text': '', 'type': '', 'category': ''}]",[],Срок просрочки 42 дня относится к категории Б....,14.04
8,catmot_09_B_disagree_only_B_consequence,"Категория Б, клиент не согласен: есть только п...",Б,Б,не согласен оплатить,не согласен оплатить,Б + не согласен: не хватает последствия катего...,False,False,True,[ст. 811 ГК РФ],[],[],"Из текста диалога видно, что просрочка 38 дней...",11.63
9,catmot_10_A_agree_B_only_period_error,"Категория A, клиент согласен: озвучено только ...",A,A,согласен оплатить,согласен оплатить,A + согласен: последствие категории Б не подхо...,False,True,False,[требование полного досрочного погашения],[{'text': 'появилось подтверждение оплаты сего...,[],Срок просрочки 14 дней попадает в категорию A....,12.84


In [ ]:
print("=== Проверка соответствия мотивации категории и статусу клиента ===")
print(f"Всего кейсов: {len(category_motivation_results_df)}")
print(f"Корректных ответов: {category_motivation_results_df['category_motivation_correct'].sum()}")
print(f"Accuracy по соответствию мотивации: {category_motivation_results_df['category_motivation_correct'].mean():.2%}")
print(f"Общее время: {category_motivation_total_time:.2f} сек.")
print(f"Среднее время на кейс: {category_motivation_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {category_motivation_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {category_motivation_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка соответствия мотивации категории и статусу клиента ===
Всего кейсов: 10
Корректных ответов: 5
Accuracy по соответствию мотивации: 50.00%
Общее время: 202.12 сек.
Среднее время на кейс: 20.21 сек.
Минимальное время: 11.63 сек.
Максимальное время: 45.15 сек.


In [ ]:
category_motivation_incorrect_df = category_motivation_results_df[category_motivation_results_df["category_motivation_correct"] == False]

category_motivation_incorrect_df[["case_id","description","expected_category",
                                  "client_category","expected_client_status",
                                  "client_status","expected_required_pattern",
                                  "expected_category_motivation_valid",
                                  "predicted","detected_consequences",
                                  "violations","explanation",]]

,case_id,description,expected_category,client_category,expected_client_status,client_status,expected_required_pattern,expected_category_motivation_valid,predicted,detected_consequences,violations,explanation
1,catmot_02_A_agree_no_consequence,"Категория A, клиент согласен: последствие не о...",A,A,согласен оплатить,согласен оплатить,A + согласен: нужно 1 последствие категории A,False,True,"[{'text': 'пожалуйста, не затягивайте', 'type'...",[],Срок просрочки 12 дней относится к категории A...
3,catmot_04_A_disagree_one_A_consequence,"Категория A, клиент не согласен: озвучено толь...",A,A,не согласен оплатить,согласен оплатить,A + не согласен: нужно 2 последствия категории A,False,True,"[{'text': 'банк вправе начислять неустойку, су...",[],Срок просрочки 24 дня относится к категории A....
5,catmot_06_B_agree_only_B_consequence,"Категория Б, клиент согласен: озвучено только ...",Б,Б,согласен оплатить,согласен оплатить,Б + согласен: нужно 1 последствие категории A,False,True,"[{'text': 'передача в работу коллекторов', 'ty...",[],Срок просрочки 36 дней относится к категории Б...
7,catmot_08_B_disagree_only_A_consequence,"Категория Б, клиент не согласен: есть только п...",Б,Б,не согласен оплатить,не согласен оплатить,Б + не согласен: не хватает последствия катего...,False,True,"[{'text': '', 'type': '', 'category': ''}]",[],Срок просрочки 42 дня относится к категории Б....
9,catmot_10_A_agree_B_only_period_error,"Категория A, клиент согласен: озвучено только ...",A,A,согласен оплатить,согласен оплатить,A + согласен: последствие категории Б не подхо...,False,True,[{'text': 'появилось подтверждение оплаты сего...,[],Срок просрочки 14 дней попадает в категорию A....


## Тестирование. Параметр 5 — есть ли исключение

### Примеры

In [ ]:
exception_detection_test_cases = [
    {
        "case_id": "exception_01_treatment",
        "description": "Исключение: клиент сообщает о лечении",
        "overdue_days": 12,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "лечение",
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете внести просроченный платёж?
Клиент: Сейчас не могу заниматься оплатой, я после инсульта и нахожусь на лечении.
Оператор: Понимаю. Желаю вам скорейшего восстановления.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_02_bankruptcy",
        "description": "Исключение: банкротство",
        "overdue_days": 18,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "банкротство",
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность. Когда сможете оплатить?
Клиент: Я сейчас прохожу процедуру банкротства, документы уже поданы юристами.
Оператор: Понял вас, информацию зафиксирую.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_03_military_service",
        "description": "Исключение: армия",
        "overdue_days": 20,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "армия",
        "dialogue": """
Оператор: Добрый день. Звоню по вопросу задолженности по договору.
Клиент: Я сейчас в армии, у меня ограниченный доступ к телефону и банковскому приложению.
Оператор: Понял. Зафиксирую информацию в обращении.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_04_prison",
        "description": "Исключение: тюрьма",
        "overdue_days": 25,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "тюрьма",
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату по просроченной задолженности?
Клиент: Я не сам плательщик, он сейчас находится в тюрьме, поэтому оплату внести не может.
Оператор: Понял, информацию передам в работу.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_05_fraud",
        "description": "Исключение: мошенничество",
        "overdue_days": 14,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "мошенничество",
        "dialogue": """
Оператор: Добрый день. По вашему договору образовалась просроченная задолженность.
Клиент: Я уже обращался в банк, там было мошенничество, деньги списали без моего участия.
Оператор: Понял вас. Информацию о мошенничестве зафиксирую.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_06_death_of_payer",
        "description": "Исключение: смерть плательщика",
        "overdue_days": 30,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "смерть плательщика",
        "dialogue": """
Оператор: Добрый день. Можем поговорить с плательщиком по вопросу задолженности?
Клиент: Плательщик умер, я родственник. Сейчас занимаемся документами.
Оператор: Примите соболезнования. Информацию зафиксирую.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_07_disability",
        "description": "Исключение: инвалидность",
        "overdue_days": 35,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "инвалидность",
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете оплатить задолженность?
Клиент: Сейчас не могу работать, у меня инвалидность, доход сильно снизился.
Оператор: Понимаю. Зафиксирую информацию.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_08_emergency",
        "description": "Исключение: ЧС",
        "overdue_days": 40,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "ЧС",
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность.
Клиент: У нас в регионе была чрезвычайная ситуация, дом пострадал, сейчас все деньги уходят на восстановление.
Оператор: Понял. Информацию о ЧС зафиксирую.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_09_no_exception_job_loss",
        "description": "Исключения нет: потеря работы не входит в список исключений ТЗ",
        "overdue_days": 16,
        "expected": False,
        "expected_exception_detected": False,
        "expected_exception_type": None,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Я потерял работу, поэтому сейчас сложно с деньгами.
Оператор: Понимаю, но при отсутствии оплаты банк вправе начислять неустойку.
Клиент: Постараюсь найти деньги позже.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "exception_10_no_exception_simple_delay",
        "description": "Исключения нет: обычная задержка зарплаты",
        "overdue_days": 22,
        "expected": False,
        "expected_exception_detected": False,
        "expected_exception_type": None,
        "dialogue": """
Оператор: Добрый день. По договору есть просроченный платёж. Когда сможете оплатить?
Клиент: Зарплату задерживают, поэтому смогу оплатить только на следующей неделе.
Оператор: В таком случае банк может начислять неустойку, сумма задолженности может увеличиваться.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
exception_results_df, exception_total_time = run_evaluation(
    test_cases=exception_detection_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
exception_results_df["expected_exception_detected"] = [
    case["expected_exception_detected"]
    for case in exception_detection_test_cases
]

exception_results_df["expected_exception_type"] = [
    case["expected_exception_type"]
    for case in exception_detection_test_cases
]

In [ ]:
exception_results_df["exception_correct"] = (
    exception_results_df["exception_detected"]
    == exception_results_df["expected_exception_detected"]
)

In [ ]:
exception_results_df[
    [
        "case_id",
        "description",
        "expected_exception_detected",
        "exception_detected",
        "exception_correct",
        "expected_exception_type",
        "expected",
        "predicted",
        "decision",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,expected_exception_detected,exception_detected,exception_correct,expected_exception_type,expected,predicted,decision,explanation,elapsed_time_sec
0,exception_01_treatment,Исключение: клиент сообщает о лечении,True,False,False,лечение,True,False,не выполнено,Срок просрочки 12 дней относится к категории A...,28.18
1,exception_02_bankruptcy,Исключение: банкротство,True,False,False,банкротство,True,True,выполнено,Срок просрочки 18 дней относится к категории A...,13.60
2,exception_03_military_service,Исключение: армия,True,False,False,армия,True,True,выполнено,Срок просрочки 20 дней попадает в категорию A....,12.71
3,exception_04_prison,Исключение: тюрьма,True,False,False,тюрьма,True,True,выполнено,Срок просрочки 25 дней (категория A). Клиент в...,12.99
4,exception_05_fraud,Исключение: мошенничество,True,False,False,мошенничество,True,True,выполнено,Срок просрочки 14 дней относится к категории A...,11.86
5,exception_06_death_of_payer,Исключение: смерть плательщика,True,False,False,смерть плательщика,True,True,выполнено,Срок просрочки 30 дней относится к категории A...,12.93
6,exception_07_disability,Исключение: инвалидность,True,False,False,инвалидность,True,True,выполнено,Срок просрочки 35 дней относится к категории Б...,12.41
7,exception_08_emergency,Исключение: ЧС,True,False,False,ЧС,True,True,выполнено,Срок просрочки 40 дней относится к категории Б...,13.18
8,exception_09_no_exception_job_loss,Исключения нет: потеря работы не входит в спис...,False,False,True,None,False,True,выполнено,Срок просрочки 16 дней относится к категории A...,12.28
9,exception_10_no_exception_simple_delay,Исключения нет: обычная задержка зарплаты,False,False,True,None,False,True,выполнено,Срок просрочки 22 дня относится к категории A....,13.02


In [ ]:
print("=== Проверка детекции исключений ===")
print(f"Всего кейсов: {len(exception_results_df)}")
print(f"Корректных ответов: {exception_results_df['exception_correct'].sum()}")
print(f"Accuracy по исключениям: {exception_results_df['exception_correct'].mean():.2%}")
print(f"Общее время: {exception_total_time:.2f} сек.")
print(f"Среднее время на кейс: {exception_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {exception_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {exception_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка детекции исключений ===
Всего кейсов: 10
Корректных ответов: 2
Accuracy по исключениям: 20.00%
Общее время: 143.21 сек.
Среднее время на кейс: 14.32 сек.
Минимальное время: 11.86 сек.
Максимальное время: 28.18 сек.


In [ ]:
exception_incorrect_df = exception_results_df[
    exception_results_df["exception_correct"] == False
]

exception_incorrect_df[
    [
        "case_id",
        "description",
        "expected_exception_detected",
        "exception_detected",
        "expected_exception_type",
        "violations",
        "explanation",
    ]
]

,case_id,description,expected_exception_detected,exception_detected,expected_exception_type,violations,explanation
0,exception_01_treatment,Исключение: клиент сообщает о лечении,True,False,лечение,[],Срок просрочки 12 дней относится к категории A...
1,exception_02_bankruptcy,Исключение: банкротство,True,False,банкротство,[],Срок просрочки 18 дней относится к категории A...
2,exception_03_military_service,Исключение: армия,True,False,армия,[],Срок просрочки 20 дней попадает в категорию A....
3,exception_04_prison,Исключение: тюрьма,True,False,тюрьма,[],Срок просрочки 25 дней (категория A). Клиент в...
4,exception_05_fraud,Исключение: мошенничество,True,False,мошенничество,[],Срок просрочки 14 дней относится к категории A...
5,exception_06_death_of_payer,Исключение: смерть плательщика,True,False,смерть плательщика,[],Срок просрочки 30 дней относится к категории A...
6,exception_07_disability,Исключение: инвалидность,True,False,инвалидность,[],Срок просрочки 35 дней относится к категории Б...
7,exception_08_emergency,Исключение: ЧС,True,False,ЧС,[],Срок просрочки 40 дней относится к категории Б...


## Тестирование. Параметр 6 — прерван диалог или нет

### Примеры

In [ ]:
interruption_detection_test_cases = [
    {
        "case_id": "interrupt_01_operator_said_goodbye",
        "description": "Диалог завершён: оператор попрощался",
        "overdue_days": 10,
        "expected": True,
        "expected_interrupted_dialog": False,
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете оплатить просроченную задолженность?
Клиент: Сегодня вечером оплачу.
Оператор: Спасибо. После поступления оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "interrupt_02_operator_no_goodbye_after_client_answer",
        "description": "Диалог прерван: оператор не попрощался после ответа клиента",
        "overdue_days": 12,
        "expected": True,
        "expected_interrupted_dialog": True,
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату?
Клиент: Возможно, завтра.
Оператор: Понял, информацию зафиксировал.
""".strip(),
    },
    {
        "case_id": "interrupt_03_client_said_goodbye_operator_not",
        "description": "Диалог прерван: клиент попрощался, оператор нет",
        "overdue_days": 15,
        "expected": True,
        "expected_interrupted_dialog": True,
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность.
Клиент: Да, сегодня оплачу.
Оператор: После оплаты звонки прекратятся.
Клиент: Хорошо, до свидания.
""".strip(),
    },
    {
        "case_id": "interrupt_04_operator_final_question_no_answer",
        "description": "Диалог прерван: оператор задал вопрос, ответа и прощания нет",
        "overdue_days": 22,
        "expected": True,
        "expected_interrupted_dialog": True,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Пока не знаю, денег нет.
Оператор: Банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Оператор: Как планируете решать вопрос?
""".strip(),
    },
    {
        "case_id": "interrupt_05_operator_goodbye_after_full_dialogue",
        "description": "Диалог завершён: есть полноценное завершение оператором",
        "overdue_days": 25,
        "expected": True,
        "expected_interrupted_dialog": False,
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете внести платёж?
Клиент: Сегодня не смогу, постараюсь завтра.
Оператор: При отсутствии оплаты банк может начислять неустойку.
Оператор: Также могут продолжаться звонки по вопросу оплаты.
Клиент: Понял, постараюсь оплатить завтра.
Оператор: Спасибо за информацию. До свидания.
""".strip(),
    },
    {
        "case_id": "interrupt_06_operator_says_thank_you_but_no_goodbye",
        "description": "Диалог прерван: оператор поблагодарил, но не попрощался",
        "overdue_days": 18,
        "expected": True,
        "expected_interrupted_dialog": True,
        "dialogue": """
Оператор: Добрый день. Подтверждаете оплату сегодня?
Клиент: Да, сегодня оплачу.
Оператор: Спасибо за информацию. После оплаты звонки прекратятся.
""".strip(),
    },
    {
        "case_id": "interrupt_07_operator_goodbye_short_dialogue",
        "description": "Диалог завершён: короткий диалог, но оператор попрощался",
        "overdue_days": 8,
        "expected": False,
        "expected_interrupted_dialog": False,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Сегодня.
Оператор: Хорошо, ожидаем платёж. До свидания.
""".strip(),
    },
    {
        "case_id": "interrupt_08_client_disconnects",
        "description": "Диалог прерван: клиент завершил разговор, оператор не попрощался",
        "overdue_days": 20,
        "expected": True,
        "expected_interrupted_dialog": True,
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность, когда сможете оплатить?
Клиент: Не буду сейчас это обсуждать.
Оператор: При отсутствии оплаты могут продолжаться звонки по задолженности.
Клиент: Всё, я отключаюсь.
""".strip(),
    },
    {
        "case_id": "interrupt_09_operator_goodbye_after_exception",
        "description": "Диалог завершён: есть исключение и оператор попрощался",
        "overdue_days": 30,
        "expected": True,
        "expected_interrupted_dialog": False,
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату по задолженности?
Клиент: Я сейчас на лечении после операции.
Оператор: Понимаю, информацию зафиксирую. Желаю восстановления.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "interrupt_10_no_operator_goodbye_after_exception",
        "description": "Диалог прерван: есть исключение, но оператор не попрощался",
        "overdue_days": 35,
        "expected": True,
        "expected_interrupted_dialog": True,
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете оплатить задолженность?
Клиент: Сейчас не могу, я прохожу процедуру банкротства.
Оператор: Понял, информацию зафиксирую.
""".strip(),
    },
]

### Тестирование

In [ ]:
interruption_results_df, interruption_total_time = run_evaluation(
    test_cases=interruption_detection_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
interruption_results_df["expected_interrupted_dialog"] = [
    case["expected_interrupted_dialog"]
    for case in interruption_detection_test_cases
]

In [ ]:
def normalize_bool(value) -> bool | None:
    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        value = value.lower().strip()

        if value == "true":
            return True

        if value == "false":
            return False

    return None

In [ ]:
interruption_results_df["predicted_interrupted_dialog"] = interruption_results_df[
    "interrupted_dialog"
].apply(normalize_bool)

interruption_results_df["interrupted_dialog_correct"] = (
    interruption_results_df["expected_interrupted_dialog"]
    == interruption_results_df["predicted_interrupted_dialog"]
)

In [ ]:
interruption_results_df[
    [
        "case_id",
        "description",
        "expected_interrupted_dialog",
        "interrupted_dialog",
        "predicted_interrupted_dialog",
        "interrupted_dialog_correct",
        "expected",
        "predicted",
        "decision",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,expected_interrupted_dialog,interrupted_dialog,predicted_interrupted_dialog,interrupted_dialog_correct,expected,predicted,decision,explanation,elapsed_time_sec
0,interrupt_01_operator_said_goodbye,Диалог завершён: оператор попрощался,False,False,False,True,True,True,выполнено,Срок просрочки 10 дней относится к категории A...,14.16
1,interrupt_02_operator_no_goodbye_after_client_...,Диалог прерван: оператор не попрощался после о...,True,False,False,False,True,True,выполнено,Оператор спрашивает о сроке оплаты; клиент отв...,13.07
2,interrupt_03_client_said_goodbye_operator_not,"Диалог прерван: клиент попрощался, оператор нет",True,False,False,False,True,True,выполнено,Срок просрочки 15 дней относится к категории A...,28.84
3,interrupt_04_operator_final_question_no_answer,"Диалог прерван: оператор задал вопрос, ответа ...",True,False,False,False,True,True,выполнено,Срок просрочки 22 дня относится к категории A....,13.56
4,interrupt_05_operator_goodbye_after_full_dialogue,Диалог завершён: есть полноценное завершение о...,False,False,False,True,True,True,выполнено,Клиент выразил согласие оплатить (постараюсь о...,13.04
5,interrupt_06_operator_says_thank_you_but_no_go...,"Диалог прерван: оператор поблагодарил, но не п...",True,False,False,False,True,True,выполнено,Оператор озвучил последствие «после оплаты зво...,11.90
6,interrupt_07_operator_goodbye_short_dialogue,"Диалог завершён: короткий диалог, но оператор ...",False,False,False,True,False,True,выполнено,,11.38
7,interrupt_08_client_disconnects,"Диалог прерван: клиент завершил разговор, опер...",True,True,True,True,True,False,не выполнено,Диалог относится к категории A (20 дней просро...,12.74
8,interrupt_09_operator_goodbye_after_exception,Диалог завершён: есть исключение и оператор по...,False,False,False,True,True,False,не выполнено,"В диалоге клиент сообщает, что находится на ле...",25.42
9,interrupt_10_no_operator_goodbye_after_exception,"Диалог прерван: есть исключение, но оператор н...",True,False,False,False,True,True,выполнено,Срок просрочки 35 дней относится к категории Б...,12.18


In [ ]:
print("=== Проверка детекции прерванного диалога ===")
print(f"Всего кейсов: {len(interruption_results_df)}")
print(f"Корректных ответов: {interruption_results_df['interrupted_dialog_correct'].sum()}")
print(f"Accuracy по прерванному диалогу: {interruption_results_df['interrupted_dialog_correct'].mean():.2%}")
print(f"Общее время: {interruption_total_time:.2f} сек.")
print(f"Среднее время на кейс: {interruption_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {interruption_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {interruption_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка детекции прерванного диалога ===
Всего кейсов: 10
Корректных ответов: 5
Accuracy по прерванному диалогу: 50.00%
Общее время: 156.32 сек.
Среднее время на кейс: 15.63 сек.
Минимальное время: 11.38 сек.
Максимальное время: 28.84 сек.


In [ ]:
interruption_incorrect_df = interruption_results_df[
    interruption_results_df["interrupted_dialog_correct"] == False
]

interruption_incorrect_df[
    [
        "case_id",
        "description",
        "expected_interrupted_dialog",
        "interrupted_dialog",
        "predicted_interrupted_dialog",
        "violations",
        "explanation",
    ]
]

,case_id,description,expected_interrupted_dialog,interrupted_dialog,predicted_interrupted_dialog,violations,explanation
1,interrupt_02_operator_no_goodbye_after_client_...,Диалог прерван: оператор не попрощался после о...,True,False,False,[],Оператор спрашивает о сроке оплаты; клиент отв...
2,interrupt_03_client_said_goodbye_operator_not,"Диалог прерван: клиент попрощался, оператор нет",True,False,False,[],Срок просрочки 15 дней относится к категории A...
3,interrupt_04_operator_final_question_no_answer,"Диалог прерван: оператор задал вопрос, ответа ...",True,False,False,[],Срок просрочки 22 дня относится к категории A....
5,interrupt_06_operator_says_thank_you_but_no_go...,"Диалог прерван: оператор поблагодарил, но не п...",True,False,False,[],Оператор озвучил последствие «после оплаты зво...
9,interrupt_10_no_operator_goodbye_after_exception,"Диалог прерван: есть исключение, но оператор н...",True,False,False,[],Срок просрочки 35 дней относится к категории Б...


## Тестирование. Параметр 7 — найдены или нет нарушения

### Примеры

In [ ]:
violation_detection_test_cases = [
    {
        "case_id": "violation_01_exact_date_and_guarantee",
        "description": "Нарушение: точная дата и гарантия передачи коллекторам",
        "overdue_days": 15,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "точная дата + гарантия последствия",
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность.
Клиент: Я смогу оплатить позже, сейчас денег нет.
Оператор: Если не оплатите до 13 апреля, мы передадим дело коллекторам.
Клиент: Я понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_02_guaranteed_future_consequence",
        "description": "Нарушение: гарантированное будущее последствие",
        "overdue_days": 18,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "гарантия будущего последствия",
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Пока не могу сказать.
Оператор: Если оплаты не будет, банк передаст задолженность коллекторам.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_03_allowed_probabilistic_wording",
        "description": "Нарушения нет: использована допустимая вероятностная формулировка",
        "overdue_days": 20,
        "expected": True,
        "expected_violation_detected": False,
        "expected_violation_type": None,
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату?
Клиент: Завтра.
Оператор: Банк может передать задолженность в работу коллекторов.
Оператор: После поступления оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_04_period_error_A_full_early_repayment",
        "description": "Нарушение: категория A, озвучено полное досрочное погашение",
        "overdue_days": 10,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "ошибка периода",
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность.
Клиент: Сегодня оплатить не смогу.
Оператор: Банк может потребовать полного досрочного погашения задолженности.
Клиент: Но просрочка небольшая.
Оператор: Информацию я обязан озвучить.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_05_period_error_A_article_811",
        "description": "Нарушение: категория A, ссылка на ст. 811 ГК РФ",
        "overdue_days": 12,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "ошибка периода",
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченный платёж?
Клиент: Возможно, через несколько дней.
Оператор: В случае неоплаты банк вправе применить ст. 811 ГК РФ.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_06_B_article_811_allowed",
        "description": "Нарушения нет: категория Б, ст. 811 ГК РФ допустима",
        "overdue_days": 38,
        "expected": True,
        "expected_violation_detected": False,
        "expected_violation_type": None,
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность.
Клиент: Сейчас оплатить не могу.
Оператор: Банк вправе начислять неустойку.
Оператор: Также возможно применение ст. 811 ГК РФ.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_07_exact_date_without_allowed_words",
        "description": "Нарушение: точная дата будущего последствия",
        "overdue_days": 22,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "точная дата последствия",
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете оплатить?
Клиент: Пока не знаю.
Оператор: Если оплаты не будет до 20 мая, 21 мая начнутся действия по передаче долга.
Клиент: Я понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_08_allowed_current_statement",
        "description": "Нарушения нет: констатация уже начисляемой неустойки",
        "overdue_days": 16,
        "expected": True,
        "expected_violation_detected": False,
        "expected_violation_type": None,
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность.
Клиент: Да, знаю.
Оператор: По договору уже начисляется неустойка, из-за чего сумма задолженности увеличивается.
Клиент: Понял, постараюсь оплатить завтра.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_09_mixed_allowed_and_forbidden",
        "description": "Нарушение: есть допустимое последствие и запрещённая гарантия",
        "overdue_days": 25,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "гарантия будущего последствия",
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Сегодня оплатить не получится.
Оператор: Банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Оператор: Также мы передадим долг коллекторам, если оплаты не будет.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "violation_10_no_violation_positive_notification",
        "description": "Нарушения нет: позитивное уведомление после оплаты",
        "overdue_days": 8,
        "expected": True,
        "expected_violation_detected": False,
        "expected_violation_type": None,
        "dialogue": """
Оператор: Добрый день. Подтверждаете оплату сегодня?
Клиент: Да, сегодня оплачу.
Оператор: После поступления оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
violation_results_df, violation_total_time = run_evaluation(
    test_cases=violation_detection_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
violation_results_df["expected_violation_detected"] = [
    case["expected_violation_detected"]
    for case in violation_detection_test_cases
]

violation_results_df["expected_violation_type"] = [
    case["expected_violation_type"]
    for case in violation_detection_test_cases
]

In [ ]:
def has_violations(value) -> bool:
    if value is None:
        return False

    if isinstance(value, list):
        return len(value) > 0

    if isinstance(value, str):
        return len(value.strip()) > 0

    return bool(value)

In [ ]:
violation_results_df["predicted_violation_detected"] = violation_results_df[
    "violations"
].apply(has_violations)

violation_results_df["violation_correct"] = (
    violation_results_df["expected_violation_detected"]
    == violation_results_df["predicted_violation_detected"]
)

In [ ]:
violation_results_df[
    [
        "case_id",
        "description",
        "expected_violation_detected",
        "predicted_violation_detected",
        "violation_correct",
        "expected_violation_type",
        "violations",
        "expected",
        "predicted",
        "decision",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,expected_violation_detected,predicted_violation_detected,violation_correct,expected_violation_type,violations,expected,predicted,decision,explanation,elapsed_time_sec
0,violation_01_exact_date_and_guarantee,Нарушение: точная дата и гарантия передачи кол...,True,False,False,точная дата + гарантия последствия,[],False,True,выполнено,Срок просрочки 15 дней попадает в категорию A....,12.22
1,violation_02_guaranteed_future_consequence,Нарушение: гарантированное будущее последствие,True,False,False,гарантия будущего последствия,[],False,True,выполнено,Срок просрочки 18 дней относится к категории A...,13.43
2,violation_03_allowed_probabilistic_wording,Нарушения нет: использована допустимая вероятн...,False,False,True,None,[],True,True,выполнено,Срок просрочки 20 дней относится к категории A...,13.28
3,violation_04_period_error_A_full_early_repayment,"Нарушение: категория A, озвучено полное досроч...",True,False,False,ошибка периода,[],False,True,выполнено,Срок просрочки 10 дней попадает в категорию A ...,13.63
4,violation_05_period_error_A_article_811,"Нарушение: категория A, ссылка на ст. 811 ГК РФ",True,False,False,ошибка периода,[],False,True,выполнено,Срок просрочки 12 дней относится к категории A...,13.22
5,violation_06_B_article_811_allowed,"Нарушения нет: категория Б, ст. 811 ГК РФ допу...",False,False,True,None,[],True,True,выполнено,Срок просрочки 38 дней относится к категории Б...,14.14
6,violation_07_exact_date_without_allowed_words,Нарушение: точная дата будущего последствия,True,False,False,точная дата последствия,[],False,True,выполнено,Срок просрочки 22 дня относится к категории A....,29.98
7,violation_08_allowed_current_statement,Нарушения нет: констатация уже начисляемой неу...,False,False,True,None,[],True,True,выполнено,Клиент согласен оплатить. Оператор сообщил о н...,12.57
8,violation_09_mixed_allowed_and_forbidden,Нарушение: есть допустимое последствие и запре...,True,False,False,гарантия будущего последствия,[],False,True,выполнено,При статусе клиента как согласен оплатить и ср...,12.49
9,violation_10_no_violation_positive_notification,Нарушения нет: позитивное уведомление после оп...,False,False,True,None,[],True,True,выполнено,Клиент выразил согласие оплатить сегодня. Опер...,12.40


In [ ]:
print("=== Проверка детекции нарушений ===")
print(f"Всего кейсов: {len(violation_results_df)}")
print(f"Корректных ответов: {violation_results_df['violation_correct'].sum()}")
print(f"Accuracy по нарушениям: {violation_results_df['violation_correct'].mean():.2%}")
print(f"Общее время: {violation_total_time:.2f} сек.")
print(f"Среднее время на кейс: {violation_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {violation_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {violation_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка детекции нарушений ===
Всего кейсов: 10
Корректных ответов: 4
Accuracy по нарушениям: 40.00%
Общее время: 147.38 сек.
Среднее время на кейс: 14.74 сек.
Минимальное время: 12.22 сек.
Максимальное время: 29.98 сек.


In [ ]:
violation_incorrect_df = violation_results_df[
    violation_results_df["violation_correct"] == False
]

violation_incorrect_df[
    [
        "case_id",
        "description",
        "expected_violation_detected",
        "predicted_violation_detected",
        "expected_violation_type",
        "violations",
        "explanation",
    ]
]

,case_id,description,expected_violation_detected,predicted_violation_detected,expected_violation_type,violations,explanation
0,violation_01_exact_date_and_guarantee,Нарушение: точная дата и гарантия передачи кол...,True,False,точная дата + гарантия последствия,[],Срок просрочки 15 дней попадает в категорию A....
1,violation_02_guaranteed_future_consequence,Нарушение: гарантированное будущее последствие,True,False,гарантия будущего последствия,[],Срок просрочки 18 дней относится к категории A...
3,violation_04_period_error_A_full_early_repayment,"Нарушение: категория A, озвучено полное досроч...",True,False,ошибка периода,[],Срок просрочки 10 дней попадает в категорию A ...
4,violation_05_period_error_A_article_811,"Нарушение: категория A, ссылка на ст. 811 ГК РФ",True,False,ошибка периода,[],Срок просрочки 12 дней относится к категории A...
6,violation_07_exact_date_without_allowed_words,Нарушение: точная дата будущего последствия,True,False,точная дата последствия,[],Срок просрочки 22 дня относится к категории A....
8,violation_09_mixed_allowed_and_forbidden,Нарушение: есть допустимое последствие и запре...,True,False,гарантия будущего последствия,[],При статусе клиента как согласен оплатить и ср...


## Тестирование. Параметр 8 — правильно ли определены озвученные последствия

### Примеры

In [ ]:
consequence_detection_test_cases = [
    {
        "case_id": "consequence_01_no_consequence",
        "description": "Последствий нет: оператор только ожидает платёж",
        "overdue_days": 10,
        "expected": False,
        "expected_consequence_types": [],
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Да, я знаю, сегодня постараюсь оплатить.
Оператор: Хорошо, ждём платёж. Пожалуйста, не затягивайте.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_02_calls",
        "description": "Последствие: звонки",
        "overdue_days": 12,
        "expected": True,
        "expected_consequence_types": ["звонки"],
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете оплатить задолженность?
Клиент: Сегодня не получится, возможно завтра.
Оператор: При отсутствии оплаты могут продолжаться звонки вам и контактным лицам.
Клиент: Понял, постараюсь оплатить завтра.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_03_penalty",
        "description": "Последствие: штрафы / неустойки",
        "overdue_days": 18,
        "expected": True,
        "expected_consequence_types": ["штрафы / неустойки"],
        "dialogue": """
Оператор: Добрый день. У вас есть просроченный платёж.
Клиент: Пока не могу оплатить.
Оператор: Банк вправе начислять штрафы и неустойку, из-за чего сумма задолженности может увеличиваться.
Клиент: Понимаю.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_04_credit_history",
        "description": "Последствие: ухудшение кредитной истории",
        "overdue_days": 14,
        "expected": True,
        "expected_consequence_types": ["ухудшение кредитной истории"],
        "dialogue": """
Оператор: Добрый день. По договору есть просроченная задолженность.
Клиент: Я понимаю, оплатить смогу вечером.
Оператор: Обращаю внимание, что просрочка может негативно повлиять на кредитную историю.
Клиент: Тогда сегодня постараюсь закрыть вопрос.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_05_employee_visit",
        "description": "Последствие: выезд сотрудника",
        "overdue_days": 20,
        "expected": True,
        "expected_consequence_types": ["выезд сотрудника"],
        "dialogue": """
Оператор: Добрый день. Когда сможете внести просроченный платёж?
Клиент: Пока не знаю, денег нет.
Оператор: При отсутствии оплаты возможен выезд сотрудника для урегулирования вопроса задолженности.
Клиент: Понял.
Оператор: Какой срок оплаты можете назвать?
Клиент: Попробую решить вопрос до конца недели.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_06_sale_of_debt",
        "description": "Последствие: продажа долга",
        "overdue_days": 24,
        "expected": True,
        "expected_consequence_types": ["продажа долга"],
        "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченной задолженности.
Клиент: Сейчас не могу закрыть всю сумму.
Оператор: При дальнейшем отсутствии оплаты банк может продать долг третьим лицам.
Клиент: Понял, постараюсь внести оплату.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_07_collectors",
        "description": "Последствие: передача в работу коллекторов",
        "overdue_days": 26,
        "expected": True,
        "expected_consequence_types": ["передача в работу коллекторов"],
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете оплатить задолженность?
Клиент: Пока точную дату назвать не могу.
Оператор: Банк может передать задолженность в работу коллекторов.
Клиент: Я понял.
Оператор: Рекомендую определить дату платежа как можно скорее.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_08_full_early_repayment",
        "description": "Последствие: требование полного досрочного погашения",
        "overdue_days": 36,
        "expected": True,
        "expected_consequence_types": ["требование полного досрочного погашения"],
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность.
Клиент: Да, понимаю, сейчас оплатить не могу.
Оператор: Банк вправе начислять неустойку.
Оператор: Также банк может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_09_article_811",
        "description": "Последствие: ст. 811 ГК РФ",
        "overdue_days": 40,
        "expected": True,
        "expected_consequence_types": ["ст. 811 ГК РФ"],
        "dialogue": """
Оператор: Добрый день. По договору сохраняется длительная просроченная задолженность.
Клиент: Сейчас оплатить не могу.
Оператор: В таком случае возможно применение ст. 811 ГК РФ.
Клиент: Понял.
Оператор: Когда сможете решить вопрос?
Клиент: Пока не знаю.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "consequence_10_client_mentions_consequence",
        "description": "Последствие озвучил клиент, оператор не озвучивал последствия",
        "overdue_days": 16,
        "expected": False,
        "expected_consequence_types": [],
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Я боюсь, что из-за просрочки испортится кредитная история.
Оператор: Понимаю ваше беспокойство. Когда сможете внести оплату?
Клиент: Завтра.
Оператор: Хорошо, ожидаем платёж.
Клиент: До свидания.
Оператор: До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
consequence_results_df, consequence_total_time = run_evaluation(
    test_cases=consequence_detection_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
consequence_results_df["expected_consequence_types"] = [
    case["expected_consequence_types"]
    for case in consequence_detection_test_cases
]

In [ ]:
def normalize_text(value) -> str:
    return str(value).lower().replace("ё", "е")

In [ ]:
def consequence_contains_all_expected(row) -> bool:
    expected_types = row["expected_consequence_types"]
    detected = normalize_text(row["detected_consequences"])

    if not expected_types:
        return detected in ["[]", "", "none", "nan"]

    return all(
        normalize_text(expected_type) in detected
        for expected_type in expected_types
    )

In [ ]:
consequence_results_df["consequences_correct"] = consequence_results_df.apply(
    consequence_contains_all_expected,
    axis=1,
)

In [ ]:
consequence_results_df[
    [
        "case_id",
        "description",
        "expected_consequence_types",
        "detected_consequences",
        "consequences_correct",
        "expected",
        "predicted",
        "decision",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,expected_consequence_types,detected_consequences,consequences_correct,expected,predicted,decision,explanation,elapsed_time_sec
0,consequence_01_no_consequence,Последствий нет: оператор только ожидает платёж,[],"[{'text': 'пожалуйста, не затягивайте', 'type'...",False,False,True,выполнено,Клиент относится к категории A (1–30 дней). Кл...,29.13
1,consequence_02_calls,Последствие: звонки,[звонки],"[{'text': 'звонки', 'type': 'звонки', 'categor...",True,True,True,выполнено,Срок просрочки 12 дней относится к категории A...,25.62
2,consequence_03_penalty,Последствие: штрафы / неустойки,[штрафы / неустойки],[{'text': 'Банк вправе начислять штрафы и неус...,True,True,False,не выполнено,Оператор указал только одно последствие (штраф...,11.17
3,consequence_04_credit_history,Последствие: ухудшение кредитной истории,[ухудшение кредитной истории],[{'text': 'просрочка может негативно повлиять ...,True,True,True,выполнено,"Клиент согласен оплатить, оператор озвучил одн...",0.72
4,consequence_05_employee_visit,Последствие: выезд сотрудника,[выезд сотрудника],[{'text': 'появляется срок оплаты до конца нед...,True,True,True,выполнено,Клиент находится в категории A (20 дней просро...,45.84
5,consequence_06_sale_of_debt,Последствие: продажа долга,[продажа долга],"[{'text': 'попытка внести оплату', 'type': 'по...",True,True,True,выполнено,"Клиент в рамках категории A говорит, что поста...",29.46
6,consequence_07_collectors,Последствие: передача в работу коллекторов,[передача в работу коллекторов],"[{'text': '', 'type': '', 'category': ''}]",False,True,True,выполнено,,60.30
7,consequence_08_full_early_repayment,Последствие: требование полного досрочного пог...,[требование полного досрочного погашения],"[{'text': 'Понял', 'type': '', 'category': ''}]",False,True,True,выполнено,"Оператор не озвучил никаких последствий, соотв...",13.29
8,consequence_09_article_811,Последствие: ст. 811 ГК РФ,[ст. 811 ГК РФ],"[{'text': 'передача в работу коллекторов', 'ty...",False,True,True,выполнено,Срок просрочки 40 дней относится к категории Б...,9.96
9,consequence_10_client_mentions_consequence,"Последствие озвучил клиент, оператор не озвучи...",[],"[{'text': 'звонки', 'type': 'последствие', 'ca...",False,False,True,выполнено,Срок просрочки 16 дней относится к категории A...,11.81


In [ ]:
print("=== Проверка извлечения озвученных последствий ===")
print(f"Всего кейсов: {len(consequence_results_df)}")
print(f"Корректных ответов: {consequence_results_df['consequences_correct'].sum()}")
print(f"Accuracy по извлечению последствий: {consequence_results_df['consequences_correct'].mean():.2%}")
print(f"Общее время: {consequence_total_time:.2f} сек.")
print(f"Среднее время на кейс: {consequence_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {consequence_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {consequence_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка извлечения озвученных последствий ===
Всего кейсов: 10
Корректных ответов: 5
Accuracy по извлечению последствий: 50.00%
Общее время: 237.34 сек.
Среднее время на кейс: 23.73 сек.
Минимальное время: 0.72 сек.
Максимальное время: 60.30 сек.


In [ ]:
consequence_incorrect_df = consequence_results_df[
    consequence_results_df["consequences_correct"] == False
]

consequence_incorrect_df[
    [
        "case_id",
        "description",
        "expected_consequence_types",
        "detected_consequences",
        "violations",
        "explanation",
    ]
]

,case_id,description,expected_consequence_types,detected_consequences,violations,explanation
0,consequence_01_no_consequence,Последствий нет: оператор только ожидает платёж,[],"[{'text': 'пожалуйста, не затягивайте', 'type'...",[],Клиент относится к категории A (1–30 дней). Кл...
6,consequence_07_collectors,Последствие: передача в работу коллекторов,[передача в работу коллекторов],"[{'text': '', 'type': '', 'category': ''}]",[],
7,consequence_08_full_early_repayment,Последствие: требование полного досрочного пог...,[требование полного досрочного погашения],"[{'text': 'Понял', 'type': '', 'category': ''}]",[],"Оператор не озвучил никаких последствий, соотв..."
8,consequence_09_article_811,Последствие: ст. 811 ГК РФ,[ст. 811 ГК РФ],"[{'text': 'передача в работу коллекторов', 'ty...",[],Срок просрочки 40 дней относится к категории Б...
9,consequence_10_client_mentions_consequence,"Последствие озвучил клиент, оператор не озвучи...",[],"[{'text': 'звонки', 'type': 'последствие', 'ca...",[],Срок просрочки 16 дней относится к категории A...


## Тестирование. Параметр 9 — правильно ли определён тип нарушения

### Примеры

In [ ]:
violation_type_test_cases = [
    {
        "case_id": "vtype_01_exact_date_collectors",
        "description": "Точная дата будущего последствия",
        "overdue_days": 15,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "точная дата наступления последствия",
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Сейчас оплатить не могу, деньги будут позже.
Оператор: Если не оплатите до 13 апреля, 14 апреля мы передадим дело коллекторам.
Клиент: Я понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_02_guaranteed_collectors",
        "description": "Гарантия будущего последствия",
        "overdue_days": 18,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "гарантия будущего последствия",
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Пока не знаю, денег нет.
Оператор: Если оплаты не будет, мы передадим задолженность коллекторам.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_03_allowed_may_collectors",
        "description": "Нарушения нет: вероятностная формулировка",
        "overdue_days": 20,
        "expected": True,
        "expected_violation_detected": False,
        "expected_violation_type": None,
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете внести оплату?
Клиент: Завтра.
Оператор: Банк может передать задолженность в работу коллекторов.
Оператор: После оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_04_period_error_full_early_repayment",
        "description": "Ошибка периода: категория A и полное досрочное погашение",
        "overdue_days": 10,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "ошибка периода",
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность.
Клиент: Сегодня оплатить не смогу.
Оператор: Банк может потребовать полного досрочного погашения задолженности.
Клиент: Но просрочка небольшая.
Оператор: Информацию я обязан озвучить.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_05_period_error_article_811",
        "description": "Ошибка периода: категория A и ст. 811 ГК РФ",
        "overdue_days": 12,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "ошибка периода",
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченный платёж?
Клиент: Возможно, через несколько дней.
Оператор: В случае неоплаты банк вправе применить ст. 811 ГК РФ.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_06_B_article_811_allowed",
        "description": "Нарушения нет: категория Б и ст. 811 ГК РФ",
        "overdue_days": 38,
        "expected": True,
        "expected_violation_detected": False,
        "expected_violation_type": None,
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность.
Клиент: Сейчас оплатить не могу.
Оператор: Банк вправе начислять неустойку.
Оператор: Также возможно применение ст. 811 ГК РФ.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_07_current_penalty_statement_allowed",
        "description": "Нарушения нет: констатация уже начисляемой неустойки",
        "overdue_days": 16,
        "expected": True,
        "expected_violation_detected": False,
        "expected_violation_type": None,
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность.
Клиент: Да, знаю.
Оператор: По договору уже начисляется неустойка, из-за чего сумма задолженности увеличивается.
Клиент: Понял, постараюсь оплатить завтра.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_08_mixed_allowed_and_guarantee",
        "description": "Есть допустимое последствие и запрещённая гарантия",
        "overdue_days": 25,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "гарантия будущего последствия",
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Сегодня оплатить не получится.
Оператор: Банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Оператор: Также мы передадим долг коллекторам, если оплаты не будет.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_09_exact_date_and_guarantee",
        "description": "Точная дата и гарантия одновременно",
        "overdue_days": 22,
        "expected": False,
        "expected_violation_detected": True,
        "expected_violation_type": "точная дата наступления последствия",
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Пока не знаю.
Оператор: Если оплаты не будет до 20 мая, 21 мая начнутся действия по передаче долга.
Клиент: Я понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "vtype_10_client_says_violation_operator_not",
        "description": "Нарушения нет: запрещённую формулировку произносит клиент, не оператор",
        "overdue_days": 14,
        "expected": True,
        "expected_violation_detected": False,
        "expected_violation_type": None,
        "dialogue": """
Оператор: Добрый день. Когда сможете внести просроченный платёж?
Клиент: Я боюсь, что завтра меня точно передадут коллекторам.
Оператор: Я таких утверждений не озвучивал. Подскажите, когда сможете оплатить?
Клиент: Завтра.
Оператор: Хорошо, ожидаем платёж. До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
violation_type_results_df, violation_type_total_time = run_evaluation(
    test_cases=violation_type_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
violation_type_results_df["expected_violation_detected"] = [
    case["expected_violation_detected"]
    for case in violation_type_test_cases
]

violation_type_results_df["expected_violation_type"] = [
    case["expected_violation_type"]
    for case in violation_type_test_cases
]

In [ ]:
def has_violations(value) -> bool:
    if value is None:
        return False

    if isinstance(value, list):
        return len(value) > 0

    if isinstance(value, str):
        return len(value.strip()) > 0

    return bool(value)

In [ ]:
def normalize_text(value) -> str:
    return str(value).lower().replace("ё", "е")

In [ ]:
def violation_type_matches(row) -> bool:
    expected_type = row["expected_violation_type"]

    if expected_type is None:
        return not has_violations(row["violations"])

    text = normalize_text(row["violations"]) + " " + normalize_text(row["explanation"])

    expected_type_norm = normalize_text(expected_type)

    if expected_type_norm == "точная дата наступления последствия":
        return "точн" in text or "дат" in text

    if expected_type_norm == "гарантия будущего последствия":
        return "гарант" in text or "утверд" in text or "будет" in text or "передадим" in text

    if expected_type_norm == "ошибка периода":
        return "ошибк" in text or "период" in text or "категори" in text or "811" in text or "досрочн" in text

    return expected_type_norm in text

In [ ]:
violation_type_results_df["predicted_violation_detected"] = violation_type_results_df[
    "violations"
].apply(has_violations)

violation_type_results_df["violation_detected_correct"] = (
    violation_type_results_df["expected_violation_detected"]
    == violation_type_results_df["predicted_violation_detected"]
)

violation_type_results_df["violation_type_correct"] = violation_type_results_df.apply(
    violation_type_matches,
    axis=1,
)

In [ ]:
violation_type_results_df[
    [
        "case_id",
        "description",
        "expected_violation_detected",
        "predicted_violation_detected",
        "violation_detected_correct",
        "expected_violation_type",
        "violations",
        "violation_type_correct",
        "expected",
        "predicted",
        "decision",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,expected_violation_detected,predicted_violation_detected,violation_detected_correct,expected_violation_type,violations,violation_type_correct,expected,predicted,decision,explanation,elapsed_time_sec
0,vtype_01_exact_date_collectors,Точная дата будущего последствия,True,False,False,точная дата наступления последствия,[],True,False,True,выполнено,Срок просрочки 15 дней относится к категории A...,12.43
1,vtype_02_guaranteed_collectors,Гарантия будущего последствия,True,False,False,гарантия будущего последствия,[],False,False,True,выполнено,Срок просрочки 18 дней относится к категории A...,61.35
2,vtype_03_allowed_may_collectors,Нарушения нет: вероятностная формулировка,False,False,True,None,[],True,True,True,выполнено,Клиент в диалоге согласен оплатить: оператор с...,11.27
3,vtype_04_period_error_full_early_repayment,Ошибка периода: категория A и полное досрочное...,True,False,False,ошибка периода,[],True,False,False,не выполнено,Срок просрочки 10 дней попадает в категорию A....,27.65
4,vtype_05_period_error_article_811,Ошибка периода: категория A и ст. 811 ГК РФ,True,False,False,ошибка периода,[],True,False,True,выполнено,Клиент выразил согласие оплатить: оператор в р...,13.07
5,vtype_06_B_article_811_allowed,Нарушения нет: категория Б и ст. 811 ГК РФ,False,False,True,None,[],True,True,True,выполнено,Срок просрочки 38 дней относится к категории Б...,13.00
6,vtype_07_current_penalty_statement_allowed,Нарушения нет: констатация уже начисляемой неу...,False,False,True,None,[],True,True,True,выполнено,"Клиент в момент фразы оператора «Понял, постар...",12.93
7,vtype_08_mixed_allowed_and_guarantee,Есть допустимое последствие и запрещённая гара...,True,False,False,гарантия будущего последствия,[],False,False,True,выполнено,Оператор обозначил одно последствие неоплаты в...,107.34
8,vtype_09_exact_date_and_guarantee,Точная дата и гарантия одновременно,True,False,False,точная дата наступления последствия,[],False,False,True,выполнено,Срок просрочки 22 дня относится к категории A....,13.23
9,vtype_10_client_says_violation_operator_not,Нарушения нет: запрещённую формулировку произн...,False,False,True,None,[],True,True,True,выполнено,Срок просрочки 14 дней относится к категории A...,43.95


In [ ]:
print("=== Проверка типа нарушения ===")
print(f"Всего кейсов: {len(violation_type_results_df)}")
print(f"Корректная детекция нарушения: {violation_type_results_df['violation_detected_correct'].mean():.2%}")
print(f"Корректный тип нарушения: {violation_type_results_df['violation_type_correct'].mean():.2%}")
print(f"Общее время: {violation_type_total_time:.2f} сек.")
print(f"Среднее время на кейс: {violation_type_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {violation_type_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {violation_type_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка типа нарушения ===
Всего кейсов: 10
Корректная детекция нарушения: 40.00%
Корректный тип нарушения: 70.00%
Общее время: 316.25 сек.
Среднее время на кейс: 31.62 сек.
Минимальное время: 11.27 сек.
Максимальное время: 107.34 сек.


In [ ]:
violation_type_incorrect_df = violation_type_results_df[
    violation_type_results_df["violation_type_correct"] == False
]

violation_type_incorrect_df[
    [
        "case_id",
        "description",
        "expected_violation_type",
        "violations",
        "explanation",
    ]
]

,case_id,description,expected_violation_type,violations,explanation
1,vtype_02_guaranteed_collectors,Гарантия будущего последствия,гарантия будущего последствия,[],Срок просрочки 18 дней относится к категории A...
7,vtype_08_mixed_allowed_and_guarantee,Есть допустимое последствие и запрещённая гара...,гарантия будущего последствия,[],Оператор обозначил одно последствие неоплаты в...
8,vtype_09_exact_date_and_guarantee,Точная дата и гарантия одновременно,точная дата наступления последствия,[],Срок просрочки 22 дня относится к категории A....


## Тестирование. Параметр 10 — правильно ли определён тип исключения

### Примеры

In [ ]:
exception_type_test_cases = [
    {
        "case_id": "extype_01_treatment_stroke",
        "description": "Исключение: лечение после инсульта",
        "overdue_days": 12,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "лечение",
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете внести просроченный платёж?
Клиент: Сейчас не могу заниматься оплатой, я после инсульта и нахожусь на лечении.
Оператор: Понимаю. Информацию зафиксирую.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "extype_02_bankruptcy",
        "description": "Исключение: банкротство",
        "overdue_days": 18,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "банкротство",
        "dialogue": """
Оператор: Добрый день. По договору есть просроченная задолженность. Когда сможете оплатить?
Клиент: Сейчас оплату обсуждать не могу, я прохожу процедуру банкротства.
Оператор: Документы по процедуре уже поданы?
Клиент: Да, юрист занимается этим вопросом.
Оператор: Понял, информацию зафиксирую. До свидания.
""".strip(),
    },
    {
        "case_id": "extype_03_military_service",
        "description": "Исключение: армия",
        "overdue_days": 20,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "армия",
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату по просроченной задолженности?
Клиент: Я сейчас в армии, у меня нет постоянного доступа к банку и телефону.
Оператор: Понял. Зафиксирую информацию в обращении.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "extype_04_prison",
        "description": "Исключение: тюрьма",
        "overdue_days": 25,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "тюрьма",
        "dialogue": """
Оператор: Добрый день. Можем поговорить с плательщиком по вопросу задолженности?
Клиент: Плательщик сейчас в тюрьме, я родственник.
Оператор: Понял. Оплату сейчас внести невозможно?
Клиент: Да, он сам оплатить не может.
Оператор: Информацию зафиксирую. До свидания.
""".strip(),
    },
    {
        "case_id": "extype_05_fraud",
        "description": "Исключение: мошенничество",
        "overdue_days": 14,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "мошенничество",
        "dialogue": """
Оператор: Добрый день. По вашему договору образовалась просроченная задолженность.
Клиент: Я уже обращался в банк, это мошенничество. Деньги списали без моего участия.
Оператор: Заявление уже подавали?
Клиент: Да, обращение зарегистрировано.
Оператор: Понял, информацию зафиксирую. До свидания.
""".strip(),
    },
    {
        "case_id": "extype_06_death_of_payer",
        "description": "Исключение: смерть плательщика",
        "overdue_days": 30,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "смерть плательщика",
        "dialogue": """
Оператор: Добрый день. Могу поговорить с плательщиком по вопросу задолженности?
Клиент: Плательщик умер, я его родственник.
Оператор: Примите соболезнования. Документы уже оформляются?
Клиент: Да, сейчас занимаемся этим.
Оператор: Информацию зафиксирую. До свидания.
""".strip(),
    },
    {
        "case_id": "extype_07_disability",
        "description": "Исключение: инвалидность",
        "overdue_days": 35,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "инвалидность",
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Сейчас не могу работать, у меня инвалидность, доход сильно снизился.
Оператор: Понимаю. Информацию нужно зафиксировать.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "extype_08_emergency",
        "description": "Исключение: ЧС",
        "overdue_days": 40,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "ЧС",
        "dialogue": """
Оператор: Добрый день. У вас длительная просроченная задолженность.
Клиент: В нашем регионе была чрезвычайная ситуация, дом пострадал, сейчас все деньги уходят на восстановление.
Оператор: Понял. Информацию о ЧС зафиксирую.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "extype_09_military_actions",
        "description": "Исключение: военные действия",
        "overdue_days": 42,
        "expected": True,
        "expected_exception_detected": True,
        "expected_exception_type": "военные действия",
        "dialogue": """
Оператор: Добрый день. Когда сможете внести оплату по задолженности?
Клиент: Сейчас не могу, я нахожусь в зоне военных действий, доступ к банку нестабильный.
Оператор: Понял вас. Информацию зафиксирую.
Клиент: Спасибо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "extype_10_no_exception_job_loss",
        "description": "Исключения нет: потеря работы не входит в список исключений",
        "overdue_days": 16,
        "expected": False,
        "expected_exception_detected": False,
        "expected_exception_type": None,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Я потерял работу, поэтому сейчас сложно с деньгами.
Оператор: Понимаю, но при отсутствии оплаты банк вправе начислять неустойку.
Клиент: Постараюсь найти деньги позже.
Оператор: До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
exception_type_results_df, exception_type_total_time = run_evaluation(
    test_cases=exception_type_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
exception_type_results_df["expected_exception_detected"] = [
    case["expected_exception_detected"]
    for case in exception_type_test_cases
]

exception_type_results_df["expected_exception_type"] = [
    case["expected_exception_type"]
    for case in exception_type_test_cases
]

In [ ]:
def normalize_text(value) -> str:
    return str(value).lower().replace("ё", "е")

In [ ]:
def exception_type_matches(row) -> bool:
    expected_type = row["expected_exception_type"]

    if expected_type is None:
        return row["exception_detected"] is False

    text = (
        normalize_text(row["explanation"])
        + " "
        + normalize_text(row["violations"])
        + " "
        + normalize_text(row["decision"])
    )

    expected_type_norm = normalize_text(expected_type)

    if expected_type_norm == "чс":
        return "чс" in text or "чрезвычайн" in text

    if expected_type_norm == "смерть плательщика":
        return "смерт" in text or "умер" in text

    if expected_type_norm == "военные действия":
        return "военн" in text

    return expected_type_norm in text

In [ ]:
exception_type_results_df["exception_detected_correct"] = (
    exception_type_results_df["exception_detected"]
    == exception_type_results_df["expected_exception_detected"]
)

exception_type_results_df["exception_type_correct"] = exception_type_results_df.apply(
    exception_type_matches,
    axis=1,
)

In [ ]:
exception_type_results_df[
    [
        "case_id",
        "description",
        "expected_exception_detected",
        "exception_detected",
        "exception_detected_correct",
        "expected_exception_type",
        "exception_type_correct",
        "expected",
        "predicted",
        "decision",
        "explanation",
        "elapsed_time_sec",
    ]
]

,case_id,description,expected_exception_detected,exception_detected,exception_detected_correct,expected_exception_type,exception_type_correct,expected,predicted,decision,explanation,elapsed_time_sec
0,extype_01_treatment_stroke,Исключение: лечение после инсульта,True,False,False,лечение,False,True,True,выполнено,Срок просрочки 12 дней попадает в категорию A....,44.17
1,extype_02_bankruptcy,Исключение: банкротство,True,False,False,банкротство,False,True,True,выполнено,Диалог не содержит озвучивания оператором каки...,11.94
2,extype_03_military_service,Исключение: армия,True,False,False,армия,False,True,True,выполнено,Срок просрочки 20 дней относится к категории A...,12.84
3,extype_04_prison,Исключение: тюрьма,True,False,False,тюрьма,False,True,True,выполнено,Срок просрочки 25 дней относится к категории A...,11.24
4,extype_05_fraud,Исключение: мошенничество,True,False,False,мошенничество,False,True,False,не выполнено,Срок просрочки 14 дней относится к категории A...,125.04
5,extype_06_death_of_payer,Исключение: смерть плательщика,True,False,False,смерть плательщика,False,True,True,выполнено,,11.72
6,extype_07_disability,Исключение: инвалидность,True,False,False,инвалидность,False,True,True,выполнено,Срок просрочки 35 дней относится к категории Б...,45.06
7,extype_08_emergency,Исключение: ЧС,True,False,False,ЧС,False,True,True,выполнено,"По информации, срок просрочки 40 дней относитс...",27.87
8,extype_09_military_actions,Исключение: военные действия,True,False,False,военные действия,False,True,True,выполнено,,11.71
9,extype_10_no_exception_job_loss,Исключения нет: потеря работы не входит в спис...,False,False,True,None,True,False,True,выполнено,Срок просрочки 16 дней относится к категории A...,12.61


In [ ]:
print("=== Проверка типа исключения ===")
print(f"Всего кейсов: {len(exception_type_results_df)}")
print(f"Корректная детекция исключения: {exception_type_results_df['exception_detected_correct'].mean():.2%}")
print(f"Корректный тип исключения: {exception_type_results_df['exception_type_correct'].mean():.2%}")
print(f"Общее время: {exception_type_total_time:.2f} сек.")
print(f"Среднее время на кейс: {exception_type_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {exception_type_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {exception_type_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка типа исключения ===
Всего кейсов: 10
Корректная детекция исключения: 10.00%
Корректный тип исключения: 10.00%
Общее время: 314.23 сек.
Среднее время на кейс: 31.42 сек.
Минимальное время: 11.24 сек.
Максимальное время: 125.04 сек.


In [ ]:
exception_type_incorrect_df = exception_type_results_df[
    exception_type_results_df["exception_type_correct"] == False
]

exception_type_incorrect_df[
    [
        "case_id",
        "description",
        "expected_exception_detected",
        "exception_detected",
        "expected_exception_type",
        "explanation",
        "violations",
    ]
]

,case_id,description,expected_exception_detected,exception_detected,expected_exception_type,explanation,violations
0,extype_01_treatment_stroke,Исключение: лечение после инсульта,True,False,лечение,Срок просрочки 12 дней попадает в категорию A....,[]
1,extype_02_bankruptcy,Исключение: банкротство,True,False,банкротство,Диалог не содержит озвучивания оператором каки...,[]
2,extype_03_military_service,Исключение: армия,True,False,армия,Срок просрочки 20 дней относится к категории A...,[]
3,extype_04_prison,Исключение: тюрьма,True,False,тюрьма,Срок просрочки 25 дней относится к категории A...,[]
4,extype_05_fraud,Исключение: мошенничество,True,False,мошенничество,Срок просрочки 14 дней относится к категории A...,[оператор не озвучил требуемые последствия нео...
5,extype_06_death_of_payer,Исключение: смерть плательщика,True,False,смерть плательщика,,[]
6,extype_07_disability,Исключение: инвалидность,True,False,инвалидность,Срок просрочки 35 дней относится к категории Б...,[]
7,extype_08_emergency,Исключение: ЧС,True,False,ЧС,"По информации, срок просрочки 40 дней относитс...",[]
8,extype_09_military_actions,Исключение: военные действия,True,False,военные действия,,[]


## Тестирование. Параметр 11 — понимание ролей «Оператор / Клиент»

### Примеры

In [ ]:
role_understanding_test_cases = [
    {
        "case_id": "role_01_operator_consequence_client_agrees",
        "description": "Оператор озвучил последствие, клиент согласился оплатить",
        "overdue_days": 10,
        "expected": True,
        "expected_client_status": "согласен оплатить",
        "expected_consequence_types": ["звонки"],
        "expected_violation_detected": False,
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Да, я знаю.
Оператор: Подтверждаете оплату сегодня?
Клиент: Да, сегодня оплачу.
Оператор: После поступления оплаты звонки по задолженности прекратятся.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "role_02_client_mentions_consequence_operator_not",
        "description": "Последствие озвучил клиент, оператор последствий не называл",
        "overdue_days": 12,
        "expected": False,
        "expected_client_status": "согласен оплатить",
        "expected_consequence_types": [],
        "expected_violation_detected": False,
        "dialogue": """
Оператор: Добрый день. Когда сможете внести просроченный платёж?
Клиент: Сегодня оплачу. Я понимаю, что иначе могут быть звонки и проблемы с кредитной историей.
Оператор: Хорошо, ожидаем поступление платежа.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "role_03_client_says_forbidden_wording_operator_not",
        "description": "Запрещённую формулировку произносит клиент, не оператор",
        "overdue_days": 14,
        "expected": False,
        "expected_client_status": "согласен оплатить",
        "expected_consequence_types": [],
        "expected_violation_detected": False,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Завтра оплачу. Просто боюсь, что 13 апреля меня точно передадут коллекторам.
Оператор: Я таких формулировок не озвучивал. Ожидаем оплату завтра.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "role_04_operator_mentions_allowed_client_mentions_extra",
        "description": "Оператор озвучил одно последствие, клиент добавил другое от себя",
        "overdue_days": 20,
        "expected": True,
        "expected_client_status": "согласен оплатить",
        "expected_consequence_types": ["штрафы / неустойки"],
        "expected_violation_detected": False,
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете оплатить просроченную задолженность?
Клиент: Завтра внесу платёж.
Оператор: При отсутствии оплаты банк вправе начислять неустойку, из-за чего сумма задолженности может увеличиваться.
Клиент: Понял. Ещё, наверное, кредитная история испортится.
Оператор: Ожидаем оплату завтра. До свидания.
""".strip(),
    },
    {
        "case_id": "role_05_operator_no_client_status",
        "description": "Есть реплики оператора, но нет явной реплики клиента с согласием или отказом",
        "overdue_days": 10,
        "expected": False,
        "expected_client_status": "unknown",
        "expected_consequence_types": ["звонки"],
        "expected_violation_detected": False,
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Оператор: После поступления оплаты звонки по задолженности прекратятся.
Оператор: Подтвердите, пожалуйста, дату оплаты.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "role_06_client_only_no_operator",
        "description": "В диалоге есть только реплики клиента, оператор отсутствует",
        "overdue_days": 15,
        "expected": False,
        "expected_client_status": "согласен оплатить",
        "expected_consequence_types": [],
        "expected_violation_detected": False,
        "dialogue": """
Клиент: Я сегодня оплачу задолженность.
Клиент: Понимаю, что иначе могут быть штрафы и звонки.
Клиент: Постараюсь закрыть вопрос до вечера.
""".strip(),
    },
    {
        "case_id": "role_07_operator_forbidden_client_agrees",
        "description": "Запрещённую формулировку произносит оператор",
        "overdue_days": 16,
        "expected": False,
        "expected_client_status": "согласен оплатить",
        "expected_consequence_types": ["передача в работу коллекторов"],
        "expected_violation_detected": True,
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность.
Клиент: Я смогу оплатить завтра.
Оператор: Если не оплатите до 13 апреля, мы передадим дело коллекторам.
Клиент: Я понял, завтра оплачу.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "role_08_client_refuses_operator_consequences",
        "description": "Клиент отказывается, оператор озвучивает два последствия",
        "overdue_days": 24,
        "expected": True,
        "expected_client_status": "не согласен оплатить",
        "expected_consequence_types": ["штрафы / неустойки", "звонки"],
        "expected_violation_detected": False,
        "dialogue": """
Оператор: Добрый день. Когда сможете внести просроченный платёж?
Клиент: Сейчас не смогу, денег нет.
Оператор: В таком случае банк вправе начислять неустойку, из-за чего сумма задолженности может увеличиваться.
Оператор: Также могут продолжаться звонки вам и контактным лицам.
Клиент: Понимаю, но сегодня оплатить не смогу.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "role_09_client_role_contains_operator_like_text",
        "description": "Клиент цитирует фразу оператора, но сам оператор её не произносит",
        "overdue_days": 18,
        "expected": False,
        "expected_client_status": "не согласен оплатить",
        "expected_consequence_types": [],
        "expected_violation_detected": False,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Мне вчера другой сотрудник сказал: «банк может начислять штрафы», но сейчас я оплатить не могу.
Оператор: Понял. Какой срок оплаты можете назвать?
Клиент: Пока никакой.
Оператор: До свидания.
""".strip(),
    },
    {
    "case_id": "role_10_client_mentions_future_consequence_operator_not",
    "description": "Клиент сам предполагает будущее последствие, оператор его не озвучивает",
    "overdue_days": 28,
    "expected": False,
    "expected_client_status": "не согласен оплатить",
    "expected_consequence_types": [],
    "expected_violation_detected": False,
    "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете оплатить просроченную задолженность?
Клиент: Пока не знаю. Я переживаю, что завтра банк передаст долг коллекторам.
Оператор: Понимаю ваше беспокойство. Сейчас мне нужно зафиксировать, можете ли вы назвать срок оплаты.
Клиент: Нет, срок назвать не могу.
Оператор: Информацию зафиксировал. До свидания.
""".strip(),
    },
]

### Тестирование

In [ ]:
role_results_df, role_total_time = run_evaluation(
    test_cases=role_understanding_test_cases,
    verbose=False,
)

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
role_results_df["expected_client_status"] = [
    case["expected_client_status"]
    for case in role_understanding_test_cases
]

role_results_df["expected_consequence_types"] = [
    case["expected_consequence_types"]
    for case in role_understanding_test_cases
]

role_results_df["expected_violation_detected"] = [
    case["expected_violation_detected"]
    for case in role_understanding_test_cases
]

In [ ]:
def normalize_text(value) -> str:
    return str(value).lower().replace("ё", "е")

In [ ]:
def normalize_client_status(status: str) -> str:
    if not isinstance(status, str):
        return "unknown"

    status = status.lower().strip()

    if "не соглас" in status:
        return "не согласен оплатить"

    if "соглас" in status:
        return "согласен оплатить"

    return "unknown"

In [ ]:
def has_violations(value) -> bool:
    if value is None:
        return False

    if isinstance(value, list):
        return len(value) > 0

    if isinstance(value, str):
        return len(value.strip()) > 0

    return bool(value)

In [ ]:
def consequences_match_expected(row) -> bool:
    expected_types = row["expected_consequence_types"]
    detected = normalize_text(row["detected_consequences"])

    if not expected_types:
        return detected in ["[]", "", "none", "nan"]

    return all(
        normalize_text(expected_type) in detected
        for expected_type in expected_types
    )

In [ ]:
role_results_df["predicted_client_status_norm"] = role_results_df[
    "client_status"
].apply(normalize_client_status)

role_results_df["client_status_correct"] = (
    role_results_df["expected_client_status"]
    == role_results_df["predicted_client_status_norm"]
)

role_results_df["consequences_role_correct"] = role_results_df.apply(
    consequences_match_expected,
    axis=1,
)

role_results_df["predicted_violation_detected"] = role_results_df[
    "violations"
].apply(has_violations)

role_results_df["violation_role_correct"] = (
    role_results_df["expected_violation_detected"]
    == role_results_df["predicted_violation_detected"]
)

role_results_df["role_understanding_correct"] = (
    role_results_df["client_status_correct"]
    & role_results_df["consequences_role_correct"]
    & role_results_df["violation_role_correct"]
)

In [ ]:
print("=== Проверка понимания ролей Оператор / Клиент ===")
print(f"Всего кейсов: {len(role_results_df)}")
print(f"Accuracy по статусу клиента: {role_results_df['client_status_correct'].mean():.2%}")
print(f"Accuracy по последствиям из реплик оператора: {role_results_df['consequences_role_correct'].mean():.2%}")
print(f"Accuracy по нарушениям из реплик оператора: {role_results_df['violation_role_correct'].mean():.2%}")
print(f"Общая accuracy по пониманию ролей: {role_results_df['role_understanding_correct'].mean():.2%}")
print(f"Общее время: {role_total_time:.2f} сек.")
print(f"Среднее время на кейс: {role_results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {role_results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {role_results_df['elapsed_time_sec'].max():.2f} сек.")

=== Проверка понимания ролей Оператор / Клиент ===
Всего кейсов: 10
Accuracy по статусу клиента: 90.00%
Accuracy по последствиям из реплик оператора: 10.00%
Accuracy по нарушениям из реплик оператора: 90.00%
Общая accuracy по пониманию ролей: 10.00%
Общее время: 194.27 сек.
Среднее время на кейс: 19.43 сек.
Минимальное время: 4.76 сек.
Максимальное время: 29.63 сек.


In [ ]:
role_incorrect_df = role_results_df[
    role_results_df["role_understanding_correct"] == False
]

role_incorrect_df[
    [
        "case_id",
        "description",
        "expected_client_status",
        "client_status",
        "expected_consequence_types",
        "detected_consequences",
        "expected_violation_detected",
        "violations",
        "explanation",
    ]
]

,case_id,description,expected_client_status,client_status,expected_consequence_types,detected_consequences,expected_violation_detected,violations,explanation
1,role_02_client_mentions_consequence_operator_not,"Последствие озвучил клиент, оператор последств...",согласен оплатить,согласен оплатить,[],"[{'text': 'звонки', 'type': 'последствие', 'ca...",False,[],Срок просрочки 12 дней относится к категории A...
2,role_03_client_says_forbidden_wording_operator...,"Запрещённую формулировку произносит клиент, не...",согласен оплатить,согласен оплатить,[],"[{'text': 'звонки', 'type': 'последствие', 'ca...",False,[],Согласие клиента на оплату зафиксировано в реп...
3,role_04_operator_mentions_allowed_client_menti...,"Оператор озвучил одно последствие, клиент доба...",согласен оплатить,согласен оплатить,[штрафы / неустойки],[{'text': 'При отсутствии оплаты банк вправе н...,False,[],Клиент согласился оплатить (завтра). Оператор ...
4,role_05_operator_no_client_status,"Есть реплики оператора, но нет явной реплики к...",unknown,,[звонки],"[{'text': '', 'type': '', 'category': ''}]",False,[],В диалоге оператор не сообщил ни одного послед...
5,role_06_client_only_no_operator,"В диалоге есть только реплики клиента, операто...",согласен оплатить,согласен оплатить,[],"[{'text': 'Понимаю, что иначе могут быть штраф...",False,[],Клиент в рамках категории A озвучил согласие о...
6,role_07_operator_forbidden_client_agrees,Запрещённую формулировку произносит оператор,согласен оплатить,согласен оплатить,[передача в работу коллекторов],"[{'text': 'звонки', 'type': 'последствие', 'ca...",True,[],"Оператор сообщил клиенту, что при неоплате до ..."
7,role_08_client_refuses_operator_consequences,"Клиент отказывается, оператор озвучивает два п...",не согласен оплатить,согласен оплатить,"[штрафы / неустойки, звонки]","[{'text': 'банк вправе начислять неустойку', '...",False,[],"Оператор ответил клиенту, что просрочка может ..."
8,role_09_client_role_contains_operator_like_text,"Клиент цитирует фразу оператора, но сам операт...",не согласен оплатить,не согласен оплатить,[],"[{'text': '', 'type': '', 'category': ''}]",False,[],Срок просрочки 18 дней попадает в категорию A ...
9,role_10_client_mentions_future_consequence_ope...,"Клиент сам предполагает будущее последствие, о...",не согласен оплатить,не согласен оплатить,[],"[{'text': 'звонки', 'type': 'последствие', 'ca...",False,[],Срок просрочки 28 дней относится к категории A...


# Ограничения и возможные улучшения

Текущий baseline предназначен для первичной проверки prompt-based подхода. Возможные улучшения:

| Направление | Идея |
|---|---|
| Prompt versions | Сравнить compact / balanced / detailed prompt |
| Few-shot | Добавить примеры сложных кейсов внутрь prompt |
| Prompt pipeline | Разделить извлечение признаков и финальную проверку |
| Deterministic checks | Вынести срок просрочки, ключевые исключения и запрещённые формулировки в код |
| Evaluation set | Собрать размеченный набор диалогов для регулярной проверки качества |
| Multi-model testing | Проверить переносимость prompt'а на разные LLM |
| Interface | При необходимости добавить простой интерфейс для ручной проверки диалогов |

### Повторный анализ ошибок

После уточнения правил по ошибке периода качество улучшилось, однако при повторных прогонах остались нестабильные ошибки.

Выявленные проблемы:

| Проблема | Пример | Что происходит |
|---|---|---|
| Ложное обнаружение последствий | `tz_02_A_agree_no_motivation` | Модель засчитывает мотивацию, хотя оператор не озвучил последствия |
| Слабая детекция запрещённых формулировок | `tz_05_A_forbidden_assertive_wording` | Модель видит последствие, но не блокирует критерий из-за точной даты и гарантии |
| Нестабильность ответа | разные прогоны | При одинаковом prompt'е отдельные кейсы могут оцениваться по-разному |
| Ролевое смешение | отдельные кейсы | Модель иногда делает выводы при неполном диалоге или отсутствии реплик клиента |

Текущие приоритеты доработки:
1. Улучшить детекцию запрещённых утвердительных формулировок.
2. Усилить правило: отсутствие последствий означает невыполнение критерия, если нет исключения.
3. Проверить разделение ролей «Оператор» / «Клиент».
4. Расширить тестовый набор по каждой целевой сущности.

In [ ]:
SYSTEM_PROMPT = """
Ты — LLM-модуль контроля качества диалогов взыскания.

Задача: строго по переданным правилам оценить выполнение блока
«Мотивация клиента на оплату».

Ограничения:
- оценивай только указанный критерий;
- не оценивай другие аспекты диалога;
- не добавляй факты, которых нет во входных данных;
- последствия засчитывай только из реплик оператора;
- верни только валидный JSON без markdown и комментариев.
""".strip()

In [ ]:
BUSINESS_RULES = {
    "categories": {
        "A": {
            "overdue_days": "1-30",
            "rules": {
                "согласен оплатить": (
                    "должно быть озвучено 1 последствие неоплаты "
                    "или констатация действующего последствия "
                    "или позитивное уведомление об отмене последствий"
                ),
                "не согласен оплатить": (
                    "должно быть озвучено 2 последствия неоплаты "
                    "или констатация действующих последствий"
                ),
            },
            "allowed_consequences": [
                "ухудшение кредитной истории",
                "штрафы",
                "неустойки",
                "звонки",
                "выезд сотрудника",
                "продажа долга",
                "передача в работу коллекторов",
            ],
        },
        "Б": {
            "overdue_days": "31-45",
            "rules": {
                "согласен оплатить": "должно быть озвучено 1 последствие из категории A",
                "не согласен оплатить": (
                    "должно быть озвучено 1 последствие из категории A "
                    "и 1 последствие из категории Б"
                ),
            },
            "additional_consequences": [
                "требование полного досрочного погашения",
                "ст. 811 ГК РФ",
                "продажа долга",
                "передача в работу коллекторов",
            ],
        },
    },
    "client_statuses": {
        "согласен оплатить": "клиент согласен оплатить в предложенный оператором срок",
        "не согласен оплатить": "клиент отказывается, торгуется или просит отсрочку",
        "note": "ответ вроде «угу» может считаться согласием, если подтверждает оплату",
    },
    "allowed_wording": ["вправе", "может", "возможно"],
    "forbidden_wording": [
        "точные даты последствий",
        "гарантии будущих последствий",
        "формулировки вида «будет», «завтра передадим»",
    ],
    "allowed_statements": [
        "точные суммы",
        "констатация уже начисляемых штрафов",
        "констатация ухудшающейся кредитной истории",
        "констатация поступающих звонков",
    ],
    "exceptions": [
        "банкротство",
        "ЧС",
        "военные действия",
        "тюрьма",
        "армия",
        "мошенничество",
        "смерть плательщика",
        "инвалидность",
        "лечение",
    ],
    "auto_pass_conditions": [
        "упомянуто исключение",
        "оператор не попрощался в конце диалога",
    ],
}

In [ ]:
OUTPUT_SCHEMA = {
    "criterion_met": "boolean",
    "decision": "выполнено | не выполнено",
    "client_category": "A | Б | unknown",
    "client_status": "согласен оплатить | не согласен оплатить | unknown",
    "exception_detected": "boolean",
    "interrupted_dialog": "boolean",
    "detected_consequences": [
        {
            "text": "string",
            "type": "string",
            "category": "A | Б",
        }
    ],
    "violations": [
        {
            "text": "string",
            "type": "string",
            "reason": "string",
        }
    ],
    "explanation": "string",
}

In [ ]:
def build_user_prompt(overdue_days: int, dialogue: str) -> str:
    payload = {
        "overdue_days": overdue_days,
        "dialogue": dialogue,
        "business_rules": BUSINESS_RULES,
        "output_schema": OUTPUT_SCHEMA,
    }

    return (
        "Оцени диалог по переданным правилам.\n"
        "Верни ответ строго в формате JSON по output_schema.\n\n"
        f"{json.dumps(payload, ensure_ascii=False, indent=2)}"
    )